# ЛР3 · lab3_00 — данные и среда: аудит, расписания MTA, свежий отложенный период, паспорт

Задания 2 и 7 (протокол: разд. 2–4, 12). **Accelerator:** нет. **Internet:** включён. Время: ~40–70 мин.

**Inputs:** датасет `VKR_PM232_Pevin_MTA`; *необязательно* — вывод ноутбука 16 ВКР (`16_hybrid_kaggle`)
с файлами `hyb_blend__val.npz`, `hyb_blend__test.npz` (прогноз гибрида ВКР для лестницы прогнозов, H6).

**Output:** модули `marl_*.py`, `env_data.npz` (+ `_meta.json`), `trips.parquet`, `durations.parquet`,
`holdout_ridership.parquet`, паспорт данных `data_passport.md`, рисунки, проверки среды.

Что делает:
1. Загружает данные ВКР и скачивает свежие почасовые входы после 24.09.2026 (отложенный период).
2. Скачивает расписания MTA с поправками 2023–2026: отправления по часам; проверяет дубликаты рейсов;
   время рейса по датам-образцам → время оборота.
3. Собирает массивы среды (спрос групп линий, расписание, календарь, прогнозы m4w / norm_level / гибрид ВКР,
   калибровка наполнения, парк, бюджет).
4. Паспорт данных: описательная статистика, проблемы качества, схема предобработки и разбиения.
5. Проверки среды: расписание даёт награду ровно −24; MPC и оракул на val.

In [ ]:
%%writefile /kaggle/working/marl_env.py
"""
marl_env — среда ЛР3: распределение частоты поездов метро Нью-Йорка между группами линий.

Формальная спецификация — report/env_spec.md, параметры и их обоснование — PROTOCOL.md (разд. 3–4).

Кратко. Эпизод = одни сутки (24 шага по часу). Агенты: 8 групп линий + координатор парка.
Каждый час агент линии выбирает множитель к расписанию MTA (0,7…1,3), координатор — часовой лимит
поездов в работе (0,85…1,15 от расписания). Защитный слой приводит запрос к допустимому: не реже
нормативного интервала, не больше пропускной способности, не больше лимита парка. Спрос на группу линий
экзогенный (входы на станциях, распределённые по линиям). Награда команды — минус обобщённые потери
времени пассажиров (ожидание + неуспевшие сесть + переполнение) в долях от расписания, минус штраф за
смену частоты, в конце суток — штраф за перерасход бюджета поездо-часов (бюджет = расписание).

Среда векторная: B суток идут параллельно, все эпизоды одной длины. Работает на torch (CPU).
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np
import torch

# =============================================================================
# Постановка
# =============================================================================
GROUPS = [  # (имя агента, маршруты) — группы по общим магистралям (trunk lines)
    ("1·2·3", ["1", "2", "3"]),
    ("4·5·6", ["4", "5", "6"]),
    ("7", ["7"]),
    ("A·C·E", ["A", "C", "E"]),
    ("B·D·F·M", ["B", "D", "F", "M"]),
    ("N·Q·R·W", ["N", "Q", "R", "W"]),
    ("J·Z·L", ["J", "Z", "L"]),
    ("G", ["G"]),
]
DIVISION = {"A": [0, 1, 2], "B": [3, 4, 5, 6, 7]}       # дивизион A (IRT) и B (BMT/IND) — задание 4
ROUTE_MAP = {"6X": "6", "7X": "7", "FX": "F"}            # экспрессы — та же магистраль
NOT_CONTROLLED = {"S", "GS", "FS", "H", "SI", "SIR"}     # челноки и Staten Island — вне задачи
ROUTE2GROUP = {r: g for g, (_, rs) in enumerate(GROUPS) for r in rs}

MULTS = np.array([0.70, 0.85, 1.00, 1.15, 1.30], dtype="float32")    # действия агента линии
COORD = np.array([0.85, 0.925, 1.00, 1.075, 1.15], dtype="float32")  # действия координатора
N_ACT = 5
A_SCHED = 2                                                        # индекс действия «как в расписании»
H = 24


@dataclass
class EnvParams:
    """Параметры среды. Все значения зафиксированы в PROTOCOL.md до запуска."""
    rho_peak: float = 0.90      # калибровка: наполнение в типичный будний час пик по расписанию = 90% норматива
    rho0: float = 0.60          # с этого наполнения пассажир начинает «платить» за тесноту
    rho_max: float = 1.35       # физический предел посадки (135% норматива); сверх — не уехали с первым поездом
    ivt: float = 10.0           # мин. эффективной поездки в зоне максимального наполнения
    alpha: float = 1.0          # штраф тесноты: доп. минуты на минуту поездки на единицу (ρ − ρ0)
    head_floor_tph: float = 3.0  # нормативный интервал ≤ 20 мин на каждую работающую линию
    fleet_spare: float = 0.05   # запас парка сверх пикового выпуска по расписанию
    lam_budget: float = 5.0     # штраф за перерасход бюджета: 1% перерасхода = 5% суточных потерь
    lam_switch: float = 0.1     # штраф за смену частоты (оборот)
    over_tol: float = 0.005     # перерасход > 0,5% бюджета — нарушение
    use_coord: bool = True      # абляция: без координатора (лимит = весь парк)
    comm: bool = True           # абляция: без обмена сообщениями между агентами
    forecast: str = "norm"      # источник прогноза в наблюдении: norm | m4w | hyb | oracle | none
    kappa_scale: float = 1.0    # стресс: сдвиг калибровки наполнения
    fleet_scale: float = 1.0    # стресс: нехватка подвижного состава
    budget_scale: float = 1.0   # стресс: урезанный бюджет поездо-часов
    groups: tuple = tuple(range(len(GROUPS)))   # задание 4: подсеть (дивизион A или B)

    def tag(self) -> str:
        base = EnvParams()
        diff = {k: v for k, v in self.__dict__.items() if getattr(base, k) != v}
        return ",".join(f"{k}={v}" for k, v in sorted(diff.items())) or "default"


# =============================================================================
# Данные среды
# =============================================================================
@dataclass
class EnvData:
    """Всё, что нужно среде, в виде массивов «сутки × час × группа». Готовит marl_data.build_env_data."""
    day: np.ndarray            # (D,) datetime64[D]
    part: np.ndarray           # (D,) train | val | test | holdout
    dem: np.ndarray            # (D, 24, G) спрос группы линий, входов в час
    dem_prev: np.ndarray       # (D, 2, G) спрос в 22:00 и 23:00 предыдущих суток
    sched: np.ndarray          # (D, 24, G) поездов в час по расписанию (среднее двух направлений)
    nlines: np.ndarray         # (D, 24, G) число работающих линий группы в этот час
    cal: np.ndarray            # (D, 24, C) календарь: sin/cos часа, sin/cos дня недели, выходной, праздник, школа
    fc: dict                   # имя → (D, 24, 24, G): прогноз из точки h на час h+k (k = 0…23), NaN за концом суток
    rtt: np.ndarray            # (G,) время оборота поезда, ч (поезд в работе = поездов в час × оборот)
    kappa: np.ndarray          # (G,) наполнение ρ = kappa · спрос / поездов в час (калибровка по train)
    scale_d: np.ndarray        # (G,) нормировка спроса (среднее по train)
    scale_s: np.ndarray        # (G,) нормировка расписания (среднее по train)
    track_cap: np.ndarray      # (G,) пропускная способность: максимум поездов в час по расписанию за весь период
    fleet_max: float           # поездов в работе одновременно: пик расписания × (1 + запас)
    flags: dict = field(default_factory=dict)   # (D,) режимы дня: праздник, плановые работы, событие…
    meta: dict = field(default_factory=dict)

    @property
    def D(self) -> int:
        return len(self.day)

    @property
    def G(self) -> int:
        return self.dem.shape[-1]

    def idx(self, part: str) -> np.ndarray:
        return np.where(self.part == part)[0]

    # --- сохранение ---
    def save(self, path: Path):
        arrs = {k: getattr(self, k) for k in ["day", "part", "dem", "dem_prev", "sched", "nlines", "cal", "rtt",
                                              "kappa", "scale_d", "scale_s", "track_cap"]}
        arrs["part"] = arrs["part"].astype("U8")
        arrs["fleet_max"] = np.array(self.fleet_max)
        for k, v in self.fc.items():
            arrs[f"fc__{k}"] = v.astype("float32")
        for k, v in self.flags.items():
            arrs[f"flag__{k}"] = v
        np.savez_compressed(path, **arrs)
        import json
        Path(str(path).replace(".npz", "_meta.json")).write_text(json.dumps(self.meta, ensure_ascii=False, indent=1,
                                                                            default=str), encoding="utf-8")

    @staticmethod
    def load(path: Path) -> "EnvData":
        import json
        z = np.load(path, allow_pickle=False)
        fc = {k[4:]: z[k] for k in z.files if k.startswith("fc__")}
        flags = {k[6:]: z[k] for k in z.files if k.startswith("flag__")}
        mp = Path(str(path).replace(".npz", "_meta.json"))
        meta = json.loads(mp.read_text(encoding="utf-8")) if mp.exists() else {}
        return EnvData(day=z["day"], part=z["part"].astype(object), dem=z["dem"], dem_prev=z["dem_prev"],
                       sched=z["sched"], nlines=z["nlines"], cal=z["cal"], fc=fc, rtt=z["rtt"], kappa=z["kappa"],
                       scale_d=z["scale_d"], scale_s=z["scale_s"], track_cap=z["track_cap"],
                       fleet_max=float(z["fleet_max"]), flags=flags, meta=meta)

    def subset_groups(self, groups) -> "EnvData":
        """Подсеть из части групп (задание 4: дивизион A / B). Парк делится пропорционально пиковому выпуску."""
        g = list(groups)
        insvc = (self.sched * self.rtt).sum(-1)
        insvc_g = (self.sched[..., g] * self.rtt[g]).sum(-1)
        share = float(np.nanmax(insvc_g) / max(np.nanmax(insvc), 1e-9))
        return replace(self, dem=self.dem[..., g], dem_prev=self.dem_prev[..., g], sched=self.sched[..., g],
                       nlines=self.nlines[..., g], fc={k: v[..., g] for k, v in self.fc.items()}, rtt=self.rtt[g],
                       kappa=self.kappa[g], scale_d=self.scale_d[g], scale_s=self.scale_s[g],
                       track_cap=self.track_cap[g], fleet_max=self.fleet_max * share,
                       meta={**self.meta, "groups": [GROUPS[i][0] for i in g]})


# =============================================================================
# Модель потерь пассажиров (одна для среды, MPC, оракула и метрик)
# =============================================================================
def hour_cost(d, n, kappa, P: EnvParams):
    """Потери пассажиро-минут за час на группе линий.

    d — входов в час, n — поездов в час, kappa — коэффициент наполнения. Возвращает словарь тензоров/массивов:
    J (всего), wait (ожидание H/2), extra (неуспевшие сесть ждут ещё интервал), crowd (теснота), rho (наполнение).
    """
    xp = torch if isinstance(d, torch.Tensor) else np
    n_safe = xp.clip(n, 1e-3, None) if xp is np else n.clamp(min=1e-3)
    rho = kappa * P.kappa_scale * d / n_safe
    wait = d * 30.0 / n_safe
    if xp is np:
        denied = d * np.clip(1.0 - P.rho_max / np.maximum(rho, 1e-6), 0.0, None)
        crowd = d * P.ivt * P.alpha * np.clip(rho - P.rho0, 0.0, None)
    else:
        denied = d * (1.0 - P.rho_max / rho.clamp(min=1e-6)).clamp(min=0.0)
        crowd = d * P.ivt * P.alpha * (rho - P.rho0).clamp(min=0.0)
    extra = denied * 60.0 / n_safe
    return {"J": wait + extra + crowd, "wait": wait, "extra": extra, "crowd": crowd, "rho": rho, "denied": denied}


def floors_caps(sched, nlines, track_cap, P: EnvParams):
    """Нижняя граница: нормативный интервал (≤ 20 мин на работающую линию), но не ниже 0,7 и не выше расписания.
    Верхняя: 1,3 расписания, но не выше пропускной способности магистрали (если расписание само выше — расписание)."""
    if isinstance(sched, torch.Tensor):
        lo = torch.minimum(sched, torch.maximum(MULTS[0] * sched, P.head_floor_tph * nlines))
        hi = torch.minimum(MULTS[-1] * sched, torch.maximum(track_cap.expand_as(sched), sched))
    else:
        lo = np.minimum(sched, np.maximum(MULTS[0] * sched, P.head_floor_tph * nlines))
        hi = np.minimum(MULTS[-1] * sched, np.maximum(track_cap, sched))
    return lo, hi


# =============================================================================
# Векторная среда
# =============================================================================
COMMON_DIM = 10
OWN_DIM = 11
MSG_DIM = 2


class MetroEnv:
    """B суток параллельно. reset(days) → obs, state; step(actions) → obs, state, reward, done, info."""

    def __init__(self, data: EnvData, P: EnvParams | None = None, device="cpu"):
        self.P = P or EnvParams()
        if tuple(self.P.groups) != tuple(range(len(GROUPS))):
            data = data.subset_groups(self.P.groups)
        self.data = data
        self.dev = device
        t = lambda a: torch.as_tensor(np.nan_to_num(np.asarray(a, dtype="float32")), device=device)
        self.G = data.G
        self.N = self.G + (1 if self.P.use_coord else 0)
        self.dem, self.dem_prev, self.sched = t(data.dem), t(data.dem_prev), t(data.sched)
        self.nlines, self.cal = t(data.nlines), t(data.cal)
        self.rtt, self.kappa = t(data.rtt), t(data.kappa)
        self.scale_d, self.scale_s = t(data.scale_d), t(data.scale_s)
        self.track_cap = t(data.track_cap)
        src = self.P.forecast
        if src == "oracle":
            fc = self._oracle_fc(data.dem)
        elif src == "none":
            fc = np.zeros_like(data.fc["norm"])
        else:
            if src not in data.fc:
                raise KeyError(f"нет прогноза '{src}' в данных: есть {list(data.fc)}")
            fc = data.fc[src]
        self.fc_nan = torch.as_tensor(np.isnan(fc).all(axis=(1, 2, 3)), device=device)   # сутки без прогноза
        self.fc = t(fc)
        # производные по суткам
        self.lo, self.hi = floors_caps(self.sched, self.nlines, self.track_cap, self.P)
        self.insvc_sched = (self.sched * self.rtt).sum(-1)                         # (D, 24) поездов в работе
        self.budget = self.insvc_sched.sum(-1) * self.P.budget_scale               # (D,) поездо-часов
        fm = float(data.fleet_max) * self.P.fleet_scale
        self.fleet_max = torch.full_like(self.insvc_sched, fm)
        if self.P.fleet_scale >= 1.0:   # расписание всегда выполнимо, если нет стресса нехватки парка
            self.fleet_max = torch.maximum(self.fleet_max, self.insvc_sched)
        J_s = hour_cost(self.dem, self.sched, self.kappa, self.P)["J"].sum(-1)    # (D, 24)
        self.J_sched_day = J_s.sum(-1).clamp(min=1.0)                              # (D,)
        self.obs_dim = COMMON_DIM + OWN_DIM + self.G * MSG_DIM + self.N
        self.state_dim = COMMON_DIM + (self.G + 1) * OWN_DIM
        self.n_act = N_ACT

    @staticmethod
    def _oracle_fc(dem):
        D, Hh, G = dem.shape
        out = np.full((D, Hh, Hh, G), np.nan, dtype="float32")
        for h in range(Hh):
            out[:, h, :Hh - h] = dem[:, h:]
        return out

    # ------------------------------------------------------------------
    def reset(self, days):
        self.b = torch.as_tensor(np.asarray(days), dtype=torch.long, device=self.dev)
        B = len(self.b)
        self.h = 0
        self.used = torch.zeros(B, device=self.dev)
        self.n_prev = self.sched[self.b, 0].clone()
        self.dev_prev = torch.zeros(B, self.G, device=self.dev)
        self.cum_dev = torch.zeros(B, self.G, device=self.dev)
        self.c_prev = torch.ones(B, device=self.dev)
        self.util_prev = torch.ones(B, device=self.dev)
        return self._obs()

    def _forecast(self, k):
        h = self.h + k
        if h > H - 1:
            return torch.zeros(len(self.b), self.G, device=self.dev)
        return self.fc[self.b, self.h, k]

    def _obs(self):
        b, h, G, B = self.b, self.h, self.G, len(self.b)
        sd, ss = self.scale_d, self.scale_s
        f0, f1, f2 = self._forecast(0), self._forecast(1), self._forecast(2)
        if h >= 2:
            dl1, dl2 = self.dem[b, h - 1], self.dem[b, h - 2]
        elif h == 1:
            dl1, dl2 = self.dem[b, 0], self.dem_prev[b, 1]
        else:
            dl1, dl2 = self.dem_prev[b, 1], self.dem_prev[b, 0]
        s_now = self.sched[b, h]
        s_next = self.sched[b, min(h + 1, H - 1)]
        kap = self.kappa * self.P.kappa_scale
        rho_hat = (kap * f0 / s_now.clamp(min=1e-3)).clamp(max=3.0)
        rho_last = (kap * dl1 / self.n_prev.clamp(min=1e-3)).clamp(max=3.0)
        m_last = self.n_prev / s_now.clamp(min=1e-3) - 1.0 if h == 0 else \
            self.n_prev / self.sched[b, h - 1].clamp(min=1e-3) - 1.0
        bud = self.budget[b]
        own = torch.stack([f0 / sd, f1 / sd, f2 / sd, dl1 / sd, dl2 / sd, s_now / ss, s_next / ss, rho_hat, rho_last,
                           m_last.clamp(-1, 1), self.cum_dev / (bud[:, None] / H)], dim=-1)          # (B, G, 11)
        sched_rem = self.insvc_sched[b, h:].sum(-1)
        slack = (bud - self.used - sched_rem) / bud
        common = torch.cat([self.cal[b, h], torch.stack([torch.full((B,), h / H, device=self.dev), slack,
                                                          self.c_prev - 1.0], -1)], -1)            # (B, 10)
        fm = self.fleet_max[b, h]
        coord = torch.stack([f0.sum(-1) / sd.sum(), rho_hat.mean(-1), rho_hat.max(-1).values,
                             self.insvc_sched[b, h] / fm, self.used / bud, slack, self.util_prev,
                             self.c_prev - 1.0, f1.sum(-1) / sd.sum(), dl1.sum(-1) / sd.sum(),
                             torch.full((B,), float(self.P.use_coord), device=self.dev)], -1)      # (B, 11)
        msg = torch.stack([rho_hat, m_last.clamp(-1, 1)], -1).reshape(B, G * MSG_DIM)
        if not self.P.comm:
            msg = torch.zeros_like(msg)
        own_all = torch.cat([own, coord[:, None]], 1) if self.P.use_coord else own                # (B, N, 11)
        eye = torch.eye(self.N, device=self.dev).expand(B, -1, -1)
        obs = torch.cat([common[:, None].expand(-1, self.N, -1), own_all,
                         msg[:, None].expand(-1, self.N, -1), eye], -1)
        state = torch.cat([common, own.reshape(B, -1), coord], -1)
        return obs, state

    # ------------------------------------------------------------------
    def apply_actions(self, actions):
        """Действия → поездов в час после защитного слоя. Возвращает n (B, G), c (B,), флаги вмешательства."""
        b, h = self.b, self.h
        s = self.sched[b, h]
        mult = torch.as_tensor(MULTS, device=self.dev)[actions[:, :self.G]]
        c = torch.as_tensor(COORD, device=self.dev)[actions[:, self.G]] if self.P.use_coord \
            else torch.full((len(b),), float("inf"), device=self.dev)
        lo, hi = self.lo[b, h], self.hi[b, h]
        req = mult * s
        n = torch.minimum(torch.maximum(req, lo), hi)
        clip_flag = (n - req).abs() > 1e-4
        fm = self.fleet_max[b, h]
        cap = torch.where(torch.isinf(c), fm, torch.minimum(c.clamp(max=1e6) * self.insvc_sched[b, h], fm))
        use = (n * self.rtt).sum(-1)
        floor_use = (lo * self.rtt).sum(-1)
        over = use > cap + 1e-4
        alpha = ((cap - floor_use) / (use - floor_use).clamp(min=1e-6)).clamp(0.0, 1.0)
        n = torch.where(over[:, None], lo + (n - lo) * alpha[:, None], n)
        return n, torch.where(torch.isinf(c), torch.ones_like(c), c), clip_flag, over, cap

    def step(self, actions):
        actions = torch.as_tensor(actions, device=self.dev, dtype=torch.long)
        b, h = self.b, self.h
        n, c, clip_flag, cap_flag, cap = self.apply_actions(actions)
        s = self.sched[b, h]
        d = self.dem[b, h]
        cost = hour_cost(d, n, self.kappa, self.P)
        th = (n * self.rtt).sum(-1)
        self.used = self.used + th
        dev = (n - s) * self.rtt
        switch = (dev - self.dev_prev).abs().sum(-1)
        bud = self.budget[b]
        jbar = self.J_sched_day[b] / H
        r = -cost["J"].sum(-1) / jbar - self.P.lam_switch * switch / (bud / H)
        self.cum_dev = self.cum_dev + dev
        self.dev_prev = dev
        self.n_prev = n
        self.c_prev = c
        self.util_prev = th / cap.clamp(min=1e-6)
        info = {"J": cost["J"].sum(-1), "wait": cost["wait"].sum(-1), "extra": cost["extra"].sum(-1),
                "crowd": cost["crowd"].sum(-1), "pax": d.sum(-1), "pax_crowded": (d * (cost["rho"] > 1.0)).sum(-1),
                "breach": (cost["rho"] > 1.0).float().sum(-1), "denied": cost["denied"].sum(-1),
                "trainh": th, "switch": switch, "clip": clip_flag.float().sum(-1), "capcut": cap_flag.float(),
                "n": n, "rho": cost["rho"], "J_g": cost["J"], "c": c}
        self.h += 1
        done = self.h >= H
        if done:
            overrun = (self.used / bud - 1.0).clamp(min=0.0)
            r = r - self.P.lam_budget * H * overrun
            info["overrun"] = overrun
            info["used_frac"] = self.used / bud
            obs, state = None, None
        else:
            obs, state = self._obs()
        return obs, state, r, done, info


# =============================================================================
# Неучащиеся политики: расписание, MPC на прогнозе, оракул
# =============================================================================
class SchedulePolicy:
    """Расписание MTA без изменений (контрольный baseline 1)."""
    name = "schedule"

    def act(self, env: MetroEnv, obs, state, deterministic=True):
        return torch.full((len(env.b), env.N), A_SCHED, dtype=torch.long)


class MPCPolicy:
    """Прогноз + оптимизация с горизонтом до конца суток (model predictive control).

    Каждый час: по прогнозу из источника `src` на оставшиеся часы выбирает множители из того же набора действий,
    что у агентов, минимизируя ту же функцию потерь при остатке бюджета и часовом лимите парка (двойственная
    декомпозиция: множитель λ на бюджет, μ_h на часы, бисекция). Исполняет решение текущего часа.
    src = 'oracle' — верхняя граница (идеальное знание спроса). Координатор держит максимум (лимит решает MPC).
    """

    def __init__(self, src="norm", name=None):
        self.src = src
        self.name = name or f"mpc_{src}"
        self._fc = None

    def _fc_for(self, env):
        if self._fc is None or self._fc[0] is not env:
            if self.src == "oracle":
                fc = MetroEnv._oracle_fc(env.data.dem)
            else:
                fc = env.data.fc[self.src]
            self._fc = (env, torch.as_tensor(np.nan_to_num(fc, nan=0.0)))
        return self._fc[1]

    @torch.no_grad()
    def act(self, env: MetroEnv, obs, state, deterministic=True):
        b, h, G = env.b, env.h, env.G
        fc = self._fc_for(env)[b, h, :H - h]                                     # (B, H', G)
        s = env.sched[b, h:]
        lo, hi = env.lo[b, h:], env.hi[b, h:]
        mult = torch.as_tensor(MULTS)
        n = torch.minimum(torch.maximum(mult * s[..., None], lo[..., None]), hi[..., None])  # (B, H', G, 5)
        C = hour_cost(fc[..., None], n, env.kappa[None, None, :, None], env.P)["J"]
        C = C + 1e-6 * C.mean() * (mult - 1.0).abs()          # при равенстве — действие ближе к расписанию
        U = n * env.rtt[None, None, :, None]
        R = (env.budget[b] - env.used)[:, None]                                 # (B, 1)
        cap = torch.minimum(float(COORD[-1]) * env.insvc_sched[b, h:], env.fleet_max[b, h:])   # как в среде при c = max
        scale = C.mean(dim=(1, 2, 3), keepdim=True).clamp(min=1.0) / U.mean(dim=(1, 2, 3), keepdim=True).clamp(min=1e-3)

        def choose(lam, mu):
            score = C + (lam[:, None, None, None] + mu[..., None, None]) * scale * U
            a = score.argmin(-1)
            return a, torch.gather(U, -1, a[..., None])[..., 0]

        def hour_fix(lam):
            mu = torch.zeros_like(cap)
            a, u = choose(lam, mu)
            bad = u.sum(-1) > cap + 1e-4
            if bad.any():
                lo_m, hi_m = torch.zeros_like(cap), torch.full_like(cap, 1e3)
                for _ in range(30):
                    mid = 0.5 * (lo_m + hi_m)
                    _, u = choose(lam, mid)
                    ok = u.sum(-1) <= cap + 1e-4
                    hi_m = torch.where(ok, mid, hi_m)
                    lo_m = torch.where(ok, lo_m, mid)
                mu = torch.where(bad, hi_m, mu)
                a, u = choose(lam, mu)
            return a, u

        lam = torch.zeros(len(b))
        a, u = hour_fix(lam)
        need = u.sum(dim=(1, 2)) > R[:, 0] + 1e-3
        if need.any():
            lo_l, hi_l = torch.full_like(lam, -4.0), torch.full_like(lam, 3.0)      # log10 λ
            for _ in range(30):
                mid = 0.5 * (lo_l + hi_l)
                _, uu = hour_fix(10 ** mid)
                ok = uu.sum(dim=(1, 2)) <= R[:, 0] + 1e-3
                hi_l = torch.where(ok, mid, hi_l)
                lo_l = torch.where(ok, lo_l, mid)
            lam = torch.where(need, 10 ** hi_l, lam)
            a, u = hour_fix(lam)
        act = torch.full((len(b), env.N), len(COORD) - 1, dtype=torch.long)
        act[:, :G] = a[:, 0]
        return act


# =============================================================================
# Прогон политики и метрики по суткам
# =============================================================================
@torch.no_grad()
def rollout(env: MetroEnv, policy, days, deterministic=True, record=False, override=None):
    """Прогоняет политику на сутках `days`. Возвращает словарь суточных метрик (numpy, по одной строке на сутки).

    override(env, actions) → actions — для анализа вклада агентов (подмена действий одного агента расписанием).
    """
    obs, state = env.reset(days)
    acc = {k: 0.0 for k in ["J", "wait", "extra", "crowd", "pax", "pax_crowded", "breach", "denied", "trainh",
                            "switch", "clip", "capcut", "reward"]}
    rec = {"n": [], "act": [], "rho": [], "J_g": [], "c": []}
    done = False
    while not done:
        a = policy.act(env, obs, state, deterministic=deterministic)
        if override is not None:
            a = override(env, a)
        obs, state, r, done, info = env.step(a)
        for k in acc:
            acc[k] = acc[k] + (r if k == "reward" else info[k])
        if record:
            for k in ["n", "rho", "J_g", "c"]:
                rec[k].append(info[k].cpu().numpy())
            rec["act"].append(np.asarray(a.cpu() if isinstance(a, torch.Tensor) else a))
    out = {k: (v.cpu().numpy() if isinstance(v, torch.Tensor) else np.asarray(v)) for k, v in acc.items()}
    out["overrun"] = info["overrun"].cpu().numpy()
    out["used_frac"] = info["used_frac"].cpu().numpy()
    out["J_sched"] = env.J_sched_day[env.b].cpu().numpy()
    out["day"] = np.asarray(days)
    if record:
        out["trace"] = {k: np.stack(v, 1) for k, v in rec.items()}       # (B, 24, ...)
    return out

In [ ]:
%%writefile /kaggle/working/marl_data.py
"""
marl_data — данные ЛР3: из датасета ВКР и расписаний MTA собирает EnvData для среды (marl_env).

Источники
1. Датасет ВКР `VKR_PM232_Pevin_MTA` (Kaggle): почасовые входы по 424 станционным комплексам, ковариаты,
   статические признаки станций (линии в названии комплекса).
2. MTA Subway Schedules 2023–2026 (data.ny.gov): расписание с поправками на плановые работы — число отправлений
   поездов с начальной станции по маршруту, направлению и часу; время рейса (для времени оборота).
3. MTA Subway Hourly Ridership 2025+ (data.ny.gov `5wq4-mkjj`): свежий отложенный период после конца датасета ВКР.
4. (необязательно) прогнозы гибридной модели ВКР `hyb_blend__{val,test}.npz` — вывод ноутбука 16 ВКР.

Всё время — местное время Нью-Йорка без часового пояса, как в ВКР.
"""

from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

from marl_env import GROUPS, ROUTE2GROUP, ROUTE_MAP, NOT_CONTROLLED, EnvData, EnvParams, H

# --- разбиение ЛР3 (PROTOCOL.md, разд. 2) — совпадает с ВКР; holdout — свежие данные после конца датасета ВКР
SPLIT = {
    "train": ("2023-01-30", "2026-03-31"),     # первые 4 недели — только история для прогноза «норма»
    "val": ("2026-04-01", "2026-06-30"),
    "test": ("2026-07-01", "2026-09-24"),
    "holdout": ("2026-09-25", "2026-12-31"),   # фактический конец — последний полный день в выгрузке MTA
}
EPS_G = 100.0            # сглаживание в логарифмах отношений (спрос группы — тысячи входов в час)
LAYOVER_H = 8 / 60       # стоянка на конечной, ч (в обе стороны оборота)
RTT_DEFAULT_H = 2.3      # время оборота, если по расписанию оценить не удалось
NY = "https://data.ny.gov"
SCHED_DS = {2023: "7pnn-mafy", 2024: "ebrw-j62c", 2025: "q9nv-uegs", 2026: "g8es-h7gb"}
RIDERSHIP_DS = "5wq4-mkjj"


# =============================================================================
# Поиск файлов на Kaggle
# =============================================================================
def walk(name: str, root: str = "/kaggle/input") -> list[Path]:
    out = []
    if os.path.isdir(root):
        for dp, _, fn in os.walk(root, followlinks=True):
            if name in fn:
                out.append(Path(dp) / name)
    return sorted(out)


def find_vkr_root() -> Path:
    env = os.environ.get("VKR_DATA_ROOT")
    if env:
        return Path(env)
    hits = [p for p in walk("split.json") if p.parent.name == "lab"]
    if hits:
        return hits[0].parent.parent
    for cand in [Path("dataset_kaggle"), Path("../dataset_kaggle"), Path("data"), Path("../data")]:
        if (cand / "lab" / "split.json").exists():
            return cand
    raise FileNotFoundError("Не найден lab/split.json: подключите датасет VKR_PM232_Pevin_MTA или задайте VKR_DATA_ROOT")


def find_file(name: str) -> Path | None:
    hits = walk(name)
    if hits:
        return hits[0]
    for cand in [Path("results"), Path("../results"), Path("/kaggle/working/results")]:
        for p in cand.glob(f"**/{name}"):
            return p
    return None


def station_lines(name: str) -> set[str]:
    """Маршруты из скобок названия комплекса: 'Hewes St (M,J)' → {'M','J'}. Как в ВКР и ЛР2."""
    groups = re.findall(r"\(([^)]*)\)", str(name))
    out = set()
    for g in groups:
        for r in g.split(","):
            r = ROUTE_MAP.get(r.strip(), r.strip())
            if re.fullmatch(r"[0-9A-Z]{1,2}", r) and r not in NOT_CONTROLLED:
                out.add(r)
    return out


def load_vkr(root: Path):
    lab = root / "lab"
    y = pd.read_parquet(lab / "ridership_wide.parquet")
    y.columns = y.columns.astype(str)
    cov = pd.read_parquet(lab / "covariates_time.parquet")
    static = pd.read_parquet(lab / "station_static.parquet")
    static["station_complex_id"] = static["station_complex_id"].astype(str)
    static = static.set_index("station_complex_id").loc[y.columns]
    return y, cov, static


# =============================================================================
# data.ny.gov (Socrata)
# =============================================================================
class SocrataError(RuntimeError):
    pass


def _get(url, params, tries=6, timeout=180):
    """GET с повторами. Ошибки 4xx (неверный запрос) не повторяются — сразу исключение с текстом ответа сервера."""
    import requests
    for i in range(tries):
        try:
            r = requests.get(url, params=params, timeout=timeout,
                             headers={"User-Agent": "ATML-Laba3 student project (Kaggle notebook)"})
        except Exception as e:  # noqa: BLE001  (сеть)
            if i == tries - 1:
                raise RuntimeError(f"{url}: {e}") from e
            time.sleep(5 * (i + 1))
            continue
        if 400 <= r.status_code < 500 and r.status_code != 429:
            raise SocrataError(f"{r.status_code} {url}: {r.text[:600]}")
        if r.ok:
            return r.json()
        if i == tries - 1:
            raise RuntimeError(f"{r.status_code} {url}: {r.text[:600]}")
        time.sleep(5 * (i + 1))


def socrata(ds, select, where, group=None, order=None, page=50000):
    rows, off = [], 0
    while True:
        p = {"$select": select, "$where": where, "$limit": page, "$offset": off}
        if group:
            p["$group"] = group
        if order:
            p["$order"] = order
        part = _get(f"{NY}/resource/{ds}.json", p)
        rows += part
        if len(part) < page:
            return pd.DataFrame(rows)
        off += page


TRIP_QUERIES = [
    # (имя, как фильтровать месяц, select, group). Порядок: от быстрой серверной агрегации к сырым строкам.
    ("agg_departure", "departure",
     "date_trunc_ymdh(departure_time) AS ts, line, direction, trip_type, count(*) AS n",
     "date_trunc_ymdh(departure_time), line, direction, trip_type"),
    ("agg_service_date", "service",
     "date_trunc_ymdh(departure_time) AS ts, line, direction, trip_type, count(*) AS n",
     "date_trunc_ymdh(departure_time), line, direction, trip_type"),
    ("agg_no_type", "service",
     "date_trunc_ymdh(departure_time) AS ts, line, direction, count(*) AS n",
     "date_trunc_ymdh(departure_time), line, direction"),
    ("raw_rows", "service", "departure_time, line, direction, trip_type", None),
]


def _trip_where(kind, lo, hi):
    col = "departure_time" if kind == "departure" else "service_date"
    return f"stop_order = 1 AND {col} >= '{lo:%Y-%m-%dT%H:%M:%S}' AND {col} < '{hi:%Y-%m-%dT%H:%M:%S}'"


def _trips_frame(df, raw):
    if not len(df):
        return df
    if raw:                                                     # сырые строки → агрегируем сами
        df["ts"] = pd.to_datetime(df["departure_time"]).dt.floor("h")
        if "trip_type" not in df:
            df["trip_type"] = "NA"
        df = df.groupby(["ts", "line", "direction", "trip_type"], dropna=False).size().rename("n").reset_index()
    if "trip_type" not in df:
        df["trip_type"] = "NA"
    return df


def pick_trip_query(log=print):
    """Пробует формы запроса на одном дне (лимит 5 строк), возвращает первую, которую принимает сервер."""
    ds = SCHED_DS[2024]
    lo, hi = pd.Timestamp("2024-03-06"), pd.Timestamp("2024-03-07")
    for name, kind, sel, grp in TRIP_QUERIES:
        p = {"$select": sel, "$where": _trip_where(kind, lo, hi), "$limit": 5}
        if grp:
            p["$group"] = grp
        try:
            rows = _get(f"{NY}/resource/{ds}.json", p)
            log(f"  форма запроса расписания «{name}»: принята, пример {rows[:1]}")
            if rows:
                return name, kind, sel, grp
        except SocrataError as e:
            log(f"  форма запроса расписания «{name}»: отклонена — {str(e)[:300]}")
    raise RuntimeError("сервер не принял ни одну форму запроса расписания — см. сообщения выше")


def download_trips(start: pd.Timestamp, end: pd.Timestamp, log=print) -> pd.DataFrame:
    """Отправления с начальной станции (stop_order = 1): час × маршрут × направление × тип рейса → число рейсов.

    Наборы разбиты по году даты обслуживания (service_date). При фильтре по service_date каждый рейс попадает
    ровно в один набор и один месяц. При фильтре по времени отправления рейс после полуночи лежит в наборе
    прошлого года, поэтому декабрь запрашивается до 06:00 1 января следующего года. Результаты суммируются."""
    name, kind, sel, grp = pick_trip_query(log)
    raw = grp is None
    parts = []
    for year, ds in SCHED_DS.items():
        a = max(start, pd.Timestamp(f"{year}-01-01"))
        b = min(end, pd.Timestamp(f"{year + 1}-01-01 06:00") if kind == "departure" else pd.Timestamp(f"{year + 1}-01-01"))
        if a >= b:
            continue
        for ms in pd.date_range(a.normalize().replace(day=1), b, freq="MS"):
            lo = max(ms, a)
            hi = min(ms + pd.offsets.MonthBegin(1), b)
            if ms.month == 12 and kind == "departure":
                hi = min(pd.Timestamp(f"{year + 1}-01-01 06:00"), b)
            if lo >= hi:
                continue
            df = _trips_frame(socrata(ds, sel, _trip_where(kind, lo, hi), group=grp), raw)
            if len(df):
                parts.append(df)
            log(f"  расписание {year} {lo:%Y-%m-%d}…{hi:%Y-%m-%d}: {len(df)} строк")
    t = pd.concat(parts, ignore_index=True)
    t["ts"] = pd.to_datetime(t["ts"])
    t["n"] = pd.to_numeric(t["n"])
    t["trip_type"] = t["trip_type"].fillna("NA").astype(str)
    t = t.groupby(["ts", "line", "direction", "trip_type"], as_index=False)["n"].sum()
    t.attrs["query"] = name
    return t


def check_trip_duplicates(dates, log=print) -> pd.DataFrame:
    """Проверка: один ли рейс = одна строка с stop_order = 1 (нет ли в наборе базового и изменённого рейса сразу)."""
    out = []
    for d in dates:
        d = pd.Timestamp(d)
        ds = SCHED_DS.get(d.year)
        if not ds:
            continue
        w = f"stop_order = 1 AND service_date = '{d:%Y-%m-%dT00:00:00}'"
        df = socrata(ds, "train_id, line, direction, departure_time, supplement_schedule_number", w)
        if not len(df):
            continue
        n_rows = len(df)
        n_uniq = df.drop_duplicates(["train_id", "line", "direction", "departure_time"]).shape[0]
        out.append({"date": d.date(), "rows": n_rows, "unique_trips": n_uniq, "dup_share": 1 - n_uniq / n_rows,
                    "supplements": df["supplement_schedule_number"].nunique()})
        log(f"  {d.date()}: строк {n_rows}, уникальных рейсов {n_uniq}")
    return pd.DataFrame(out)


def download_durations(dates, log=print) -> pd.DataFrame:
    """Время рейса от первой до последней остановки (по выбранным датам) — для времени оборота."""
    parts = []
    for d in dates:
        d = pd.Timestamp(d)
        ds = SCHED_DS.get(d.year)
        if not ds:
            continue
        w = f"service_date = '{d:%Y-%m-%dT00:00:00}'"
        df = socrata(ds, "line, direction, train_id, min(departure_time) AS t0, max(arrival_time) AS t1", w,
                     group="line, direction, train_id")
        if len(df):
            df["date"] = d
            parts.append(df)
        log(f"  рейсы {d.date()}: {len(df)}")
    t = pd.concat(parts, ignore_index=True)
    t["t0"], t["t1"] = pd.to_datetime(t["t0"]), pd.to_datetime(t["t1"])
    t["run_h"] = (t["t1"] - t["t0"]).dt.total_seconds() / 3600
    return t[(t["run_h"] > 0.05) & (t["run_h"] < 4)]


def download_ridership(start: pd.Timestamp, end: pd.Timestamp, stations: list[str], log=print) -> pd.DataFrame:
    """Почасовые входы по комплексам (как download_mta.py ВКР) → широкая таблица в порядке станций ВКР."""
    parts = []
    for a in pd.date_range(start, end, freq="7D"):
        b = min(a + pd.Timedelta(days=7), end)
        w = (f"transit_mode = 'subway' AND transit_timestamp >= '{a:%Y-%m-%dT%H:%M:%S}' "
             f"AND transit_timestamp < '{b:%Y-%m-%dT%H:%M:%S}'")
        df = socrata(RIDERSHIP_DS, "transit_timestamp, station_complex_id, sum(ridership) AS ridership", w,
                     group="transit_timestamp, station_complex_id")
        log(f"  входы {a:%Y-%m-%d}…{b:%Y-%m-%d}: {len(df)} строк")
        if len(df):
            parts.append(df)
    if not parts:
        return pd.DataFrame(columns=stations)
    t = pd.concat(parts, ignore_index=True)
    t["transit_timestamp"] = pd.to_datetime(t["transit_timestamp"])
    t["ridership"] = pd.to_numeric(t["ridership"])
    w = t.pivot_table(index="transit_timestamp", columns="station_complex_id", values="ridership", aggfunc="sum")
    w.columns = w.columns.astype(str)
    return w.reindex(columns=stations).fillna(0.0)


# =============================================================================
# Сборка массивов среды
# =============================================================================
def full_grid(y: pd.DataFrame, log=print) -> tuple[pd.DataFrame, int]:
    """Полная сетка часов (весенний перевод часов — 02:00 нет; осенний — сдвоенный 01:00 уже сложен MTA)."""
    idx = pd.date_range(y.index.min().normalize(), y.index.max().normalize() + pd.Timedelta(hours=23), freq="h")
    miss = int((~idx.isin(y.index)).sum())
    y = y.reindex(idx).interpolate(limit=6, limit_direction="both").fillna(0.0).clip(lower=0)
    return y, miss


def incidence(lines: list[set]) -> np.ndarray:
    """(S, G): 1, если на станции останавливается маршрут группы."""
    A = np.zeros((len(lines), len(GROUPS)), dtype="float32")
    for s, ls in enumerate(lines):
        for r in ls:
            if r in ROUTE2GROUP:
                A[s, ROUTE2GROUP[r]] = 1.0
    return A


def schedule_hourly(trips: pd.DataFrame, index: pd.DatetimeIndex, log=print):
    """Поездов в час по группам (среднее двух направлений) и число работающих линий; + журнал маршрутов."""
    t = trips.copy()
    t["route"] = t["line"].astype(str).str.strip().str.upper().map(lambda r: ROUTE_MAP.get(r, r))
    unknown = sorted(set(t["route"]) - set(ROUTE2GROUP) - NOT_CONTROLLED)
    t["g"] = t["route"].map(ROUTE2GROUP)
    t = t.dropna(subset=["g"])
    t["g"] = t["g"].astype(int)
    tt = t.groupby(["ts", "g", "route"], as_index=False)["n"].sum()
    tph = tt.groupby(["ts", "g"])["n"].sum().unstack(fill_value=0.0) / 2.0                  # два направления
    nl = (tt[tt["n"] > 0].groupby(["ts", "g"])["route"].nunique().unstack(fill_value=0))
    S = tph.reindex(index=index, columns=range(len(GROUPS)), fill_value=0.0).to_numpy("float32")
    L = nl.reindex(index=index, columns=range(len(GROUPS)), fill_value=0).to_numpy("float32")
    return S, L, unknown


def day_type(days: pd.DatetimeIndex, holiday: np.ndarray) -> np.ndarray:
    """0 — будни, 1 — суббота, 2 — воскресенье или праздник."""
    dt = np.where(days.dayofweek == 5, 1, np.where(days.dayofweek == 6, 2, 0))
    return np.where(holiday > 0, 2, dt)


def fill_schedule_gaps(S, L, days, dtype, is_train, log=print):
    """Сутки, где расписания нет совсем (вне наборов или сбой выгрузки), — типичное расписание по типу дня из train."""
    D = len(days)
    S3, L3 = S.reshape(D, H, -1).copy(), L.reshape(D, H, -1).copy()
    empty = S3.sum(axis=(1, 2)) < 1
    typ_S = np.stack([np.median(S3[(dtype == k) & is_train & ~empty], axis=0) for k in range(3)])
    typ_L = np.stack([np.median(L3[(dtype == k) & is_train & ~empty], axis=0) for k in range(3)])
    S3[empty] = typ_S[dtype[empty]]
    L3[empty] = typ_L[dtype[empty]]
    return S3, L3, empty, typ_S


def group_demand(Y: np.ndarray, A: np.ndarray, S: np.ndarray):
    """Входы станции делятся между группами её маршрутов пропорционально поездам в час (расписание известно заранее).
    Станция, где в этот час не работает ни одна группа, выпадает (челноки, автобусы на время работ) — доля в журнал."""
    den = S @ A.T                                                   # (T, S)
    z = np.where(den > 0, Y / np.maximum(den, 1e-9), 0.0)
    d = S * (z @ A)                                                 # (T, G)
    lost = float(Y[den <= 0].sum() / max(Y.sum(), 1.0))
    no_group = float(Y[:, A.sum(1) == 0].sum() / max(Y.sum(), 1.0))
    return d.astype("float32"), lost, no_group


def weekly_norm(y: np.ndarray) -> np.ndarray:
    """(T, G) медиана того же часа недели за 4 прошлые недели (как norm в ВКР)."""
    T = len(y)
    st = np.full((4, T) + y.shape[1:], np.nan, dtype="float32")
    for k in range(1, 5):
        st[k - 1, 168 * k:] = y[:-168 * k]
    with np.errstate(all="ignore"):
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            return np.nanmedian(st, axis=0)


def level_feats(y, nm, t):
    out = []
    for w in (3, 24):
        idx = t[:, None] - np.arange(1, w + 1)[None, :]
        ok = (idx >= 0).all(1)
        idx = np.clip(idx, 0, None)
        ys, ns = y[idx].sum(1), np.nan_to_num(nm[idx]).sum(1)
        r = np.log((ys + EPS_G * w) / (ns + EPS_G * w))
        out.append(np.where(ok[:, None], r, 0.0))
    return np.stack(out, -1)                                          # (O, G, 2)


def build_forecasts(y: np.ndarray, D: int, train_end_t: int, start_t: int = 24 * 30):
    """Прогнозы на группу линий из каждой точки (сутки d, час h) на часы h…23: m4w и norm_level.

    m4w — медиана 4 недель; norm — m4w × поправка уровня по последним 3 и 24 ч, коэффициенты на каждый
    горизонт k = 0…23 по train (как norm_level в ВКР, src/baselines.py). Только часы < точки прогноза."""
    nm = weekly_norm(y)
    G = y.shape[1]
    # обучение коэффициентов: точки прогноза в train, каждые 5 часов
    t_fit = np.arange(start_t, train_end_t - H, 5)
    X = level_feats(y, nm, t_fit)
    coef = np.zeros((H, 3))
    for k in range(H):
        tt = t_fit + k
        ok = ~np.isnan(nm[tt]) & (nm[tt] > 5 * EPS_G / 100)
        tgt = np.log((y[tt] + EPS_G) / (np.nan_to_num(nm[tt]) + EPS_G))
        A_ = np.c_[np.ones(ok.sum()), X[ok]]
        coef[k] = np.linalg.lstsq(A_, tgt[ok], rcond=None)[0]
    fc_m4w = np.full((D, H, H, G), np.nan, dtype="float32")
    fc_norm = np.full((D, H, H, G), np.nan, dtype="float32")
    t_all = np.arange(D * H)
    Xa = level_feats(y, nm, t_all).reshape(D, H, G, 2)
    nm3 = nm.reshape(D, H, G)
    for h in range(H):
        for k in range(H - h):
            base = nm3[:, h + k]                                           # (D, G)
            adj = coef[k, 0] + Xa[:, h] @ coef[k, 1:]
            fc_m4w[:, h, k] = base
            fc_norm[:, h, k] = np.clip((base + EPS_G) * np.exp(adj) - EPS_G, 0, None)
    return {"m4w": fc_m4w, "norm": fc_norm}, coef


def aggregate_station_forecast(origins: pd.DatetimeIndex, yhat: np.ndarray, A: np.ndarray, S_hourly: np.ndarray,
                               grid: pd.DatetimeIndex, day0: pd.Timestamp, D: int) -> np.ndarray:
    """Прогноз ВКР по станциям (O, 24, N) → по группам линий в формате (D, 24, 24, G)."""
    pos = pd.Series(np.arange(len(grid)), index=grid)
    out = np.full((D, H, H, A.shape[1]), np.nan, dtype="float32")
    ok = origins.isin(grid)
    origins, yhat = origins[ok], yhat[ok]
    t0 = pos[origins].to_numpy()
    for k in range(H):
        tk = np.clip(t0 + k, 0, len(grid) - 1)
        s = S_hourly[tk]                                         # (O, G)
        den = s @ A.T
        z = np.where(den > 0, yhat[:, k] / np.maximum(den, 1e-9), 0.0)
        dg = s * (z @ A)
        di = (origins.normalize() - day0).days.to_numpy()
        hh = origins.hour.to_numpy()
        keep = (hh + k < H) & (di >= 0) & (di < D)
        out[di[keep], hh[keep], k] = dg[keep]
    return out


def calendar(days: pd.DatetimeIndex, holiday: np.ndarray, school: np.ndarray) -> np.ndarray:
    D = len(days)
    hrs = np.arange(H)
    dow = days.dayofweek.to_numpy()
    cal = np.zeros((D, H, 7), dtype="float32")
    cal[..., 0] = np.sin(2 * np.pi * hrs / H)[None]
    cal[..., 1] = np.cos(2 * np.pi * hrs / H)[None]
    cal[..., 2] = np.sin(2 * np.pi * dow / 7)[:, None]
    cal[..., 3] = np.cos(2 * np.pi * dow / 7)[:, None]
    cal[..., 4] = (dow >= 5)[:, None]
    cal[..., 5] = holiday[:, None]
    cal[..., 6] = school[:, None]
    return cal


def us_holidays_fallback(days: pd.DatetimeIndex) -> np.ndarray:
    """Праздники для дат вне ковариат ВКР (отложенный период): федеральные + выборы (школы закрыты)."""
    try:
        import holidays
        hs = holidays.country_holidays("US", subdiv="NY", years=sorted(set(days.year)))
        return np.array([d.date() in hs for d in days], dtype="float32")
    except Exception:  # noqa: BLE001
        fixed = {"2026-10-12", "2026-11-03", "2026-11-11", "2026-11-26", "2026-12-25"}
        return np.array([d.strftime("%Y-%m-%d") in fixed for d in days], dtype="float32")


def build_env_data(y: pd.DataFrame, cov: pd.DataFrame, static: pd.DataFrame, trips: pd.DataFrame,
                   durations: pd.DataFrame | None, hyb: dict | None = None, P: EnvParams | None = None,
                   split: dict | None = None, log=print) -> tuple[EnvData, dict]:
    """Главная функция подготовки. Возвращает EnvData и паспорт (словарь с цифрами для отчёта)."""
    P = P or EnvParams()
    split = split or SPLIT
    pas: dict = {}
    y, miss = full_grid(y, log)
    grid = y.index
    days = pd.DatetimeIndex(grid[::H])
    D = len(days)
    pas["hours_filled"] = miss
    lines = [station_lines(n) for n in static["station_complex"]]
    A = incidence(lines)
    pas["stations"] = len(lines)
    pas["stations_no_group"] = int((A.sum(1) == 0).sum())
    pas["stations_no_group_names"] = list(static.loc[A.sum(1) == 0, "station_complex"].astype(str))[:30]
    # части
    part = np.full(D, "", dtype=object)
    for name, (a, b) in split.items():
        part[(days >= pd.Timestamp(a)) & (days <= pd.Timestamp(b))] = name
    is_train = part == "train"
    # календарь
    covd = cov.reindex(grid)
    hol_cov = covd["is_holiday"].to_numpy() if "is_holiday" in covd else np.full(len(grid), np.nan)
    hol = pd.Series(hol_cov).groupby(np.arange(len(grid)) // H).max().to_numpy().astype("float64").copy()
    sch_cov = covd["is_school_day"].to_numpy() if "is_school_day" in covd else np.full(len(grid), np.nan)
    sch = pd.Series(sch_cov).groupby(np.arange(len(grid)) // H).max().to_numpy().astype("float64").copy()
    nocov = np.isnan(hol)
    if nocov.any():
        hol[nocov] = us_holidays_fallback(days[nocov])
        wk = days.dayofweek < 5
        sch[nocov] = (wk & (hol == 0) & ~((days.month == 12) & (days.day >= 24)))[nocov]
    pas["days_without_vkr_covariates"] = int(nocov.sum())
    hol = np.nan_to_num(hol).astype("float32")
    sch = np.nan_to_num(sch).astype("float32")
    dtype = day_type(days, hol)
    # расписание
    S_h, L_h, unknown = schedule_hourly(trips, grid, log)
    S3, L3, empty, typ_S = fill_schedule_gaps(S_h, L_h, days, dtype, is_train, log)
    pas["routes_unknown"] = unknown
    pas["days_schedule_filled"] = int(empty.sum())
    pas["days_schedule_filled_list"] = [str(d.date()) for d in days[empty]][:50]
    S_hourly = S3.reshape(D * H, -1)
    # спрос групп
    Y = y.to_numpy("float32")
    dg, lost, no_group = group_demand(Y, A, S_hourly)
    pas["share_entries_no_group"] = no_group
    pas["share_entries_no_service"] = lost
    dem = dg.reshape(D, H, -1)
    dem_prev = np.concatenate([dem[:1, :2] * 0 + dem[:1, -2:], dem[:-1, -2:]], 0)
    # прогнозы
    t_end = int(np.where(is_train)[0].max() + 1) * H
    fc, coef = build_forecasts(dg, D, t_end)
    if hyb:
        for nm_, (orig, yh) in hyb.items():
            fc[nm_] = aggregate_station_forecast(pd.DatetimeIndex(orig), yh, A, S_hourly, grid, days[0], D)
    # время оборота
    rtt = np.full(len(GROUPS), RTT_DEFAULT_H, dtype="float32")
    rtt_src = ["по умолчанию"] * len(GROUPS)
    if durations is not None and len(durations):
        du = durations.copy()
        du["route"] = du["line"].astype(str).str.upper().map(lambda r: ROUTE_MAP.get(r, r))
        du["g"] = du["route"].map(ROUTE2GROUP)
        du = du.dropna(subset=["g"])
        per_route = du.groupby(["g", "route"]).agg(run=("run_h", "median"), n=("run_h", "size")).reset_index()
        for g, grp in per_route.groupby("g"):
            run = float(np.average(grp["run"], weights=grp["n"]))
            rtt[int(g)] = 2 * (run + LAYOVER_H)
            rtt_src[int(g)] = f"расписание: {len(grp)} маршрутов, {int(grp['n'].sum())} рейсов"
    pas["rtt_h"] = dict(zip([g for g, _ in GROUPS], np.round(rtt, 2).tolist()))
    pas["rtt_source"] = dict(zip([g for g, _ in GROUPS], rtt_src))
    # калибровка наполнения: в типичный будний час пик (train) по расписанию ρ = rho_peak
    wk = is_train & (dtype == 0)
    med_d = np.median(dem[wk], axis=0)                                  # (24, G)
    pk = med_d.argmax(0)
    d_pk = med_d[pk, np.arange(len(GROUPS))]
    n_pk = np.median(S3[wk], axis=0)[pk, np.arange(len(GROUPS))]
    kappa = (P.rho_peak * n_pk / np.maximum(d_pk, 1.0)).astype("float32")
    pas["peak_hour"] = dict(zip([g for g, _ in GROUPS], pk.tolist()))
    pas["peak_demand"] = dict(zip([g for g, _ in GROUPS], np.round(d_pk).tolist()))
    pas["peak_tph"] = dict(zip([g for g, _ in GROUPS], np.round(n_pk, 1).tolist()))
    scale_d = dem[is_train].mean(axis=(0, 1)).astype("float32")
    scale_s = S3[is_train].mean(axis=(0, 1)).astype("float32")
    track_cap = S3.max(axis=(0, 1)).astype("float32")
    insvc = (S3 * rtt).sum(-1)
    fleet_max = float(np.max(insvc[is_train]) * (1 + P.fleet_spare))
    pas["fleet_max_trains"] = round(fleet_max, 1)
    pas["budget_trainh_median_weekday"] = float(np.median(insvc[is_train & (dtype == 0)].sum(-1)))
    # режимы дня
    typ = typ_S[dtype]                                                  # (D, 24, G)
    low = (S3 < 0.8 * typ) & (typ >= 2)
    planned = (low.sum(1) >= 3).any(-1)
    ev = covd["is_city_event"].to_numpy() if "is_city_event" in covd else np.zeros(len(grid))
    ev_day = pd.Series(np.nan_to_num(ev)).groupby(np.arange(len(grid)) // H).max().to_numpy() > 0
    flags = {"holiday": hol > 0, "weekend": days.dayofweek.to_numpy() >= 5, "planned_works": planned,
             "city_event": ev_day, "schedule_filled": empty, "daytype": dtype}
    # качество прогнозов (WAPE, группа × час)
    q = {}
    for nm_, f in fc.items():
        for pp in ["train", "val", "test", "holdout"]:
            m = part == pp
            if not m.any():
                continue
            ff = f[m]
            # горизонт k = 0 (ближайший час) по всем точкам прогноза и все горизонты
            k0 = ff[:, :, 0]
            ok = ~np.isnan(k0)
            if not ok.any():
                continue
            q[f"{nm_}|{pp}|h1"] = float(np.abs(k0[ok] - dem[m][ok]).sum() / max(dem[m][ok].sum(), 1))
            allk = []
            for h in range(H):
                fk = ff[:, h, :H - h]
                tk = dem[m][:, h:]
                ok = ~np.isnan(fk)
                allk.append((np.abs(fk[ok] - tk[ok]).sum(), tk[ok].sum()))
            num = sum(a for a, _ in allk)
            den = sum(b for _, b in allk)
            q[f"{nm_}|{pp}|all"] = float(num / max(den, 1))
    pas["forecast_wape"] = q
    pas["norm_coef"] = coef.round(4).tolist()
    pas["days"] = {pp: int((part == pp).sum()) for pp in split}
    pas["period"] = [str(days[0].date()), str(days[-1].date())]
    data = EnvData(day=days.values.astype("datetime64[D]"), part=part, dem=dem, dem_prev=dem_prev.astype("float32"),
                   sched=S3.astype("float32"), nlines=L3.astype("float32"),
                   cal=calendar(days, hol, sch), fc=fc, rtt=rtt, kappa=kappa, scale_d=scale_d, scale_s=scale_s,
                   track_cap=track_cap, fleet_max=fleet_max, flags=flags,
                   meta={"groups": [g for g, _ in GROUPS], "passport": pas, "params": P.__dict__})
    return data, pas

In [ ]:
%%writefile /kaggle/working/marl_algos.py
"""
marl_algos — 8 MARL-алгоритмов и одноагентный PPO для среды marl_env (PyTorch, CPU).

Все алгоритмы реализованы в одном файле по одной схеме, чтобы сравнение было честным (PROTOCOL.md, разд. 5):
одна среда, одни наблюдения, один бюджет взаимодействий (шагов среды), одни seed, одна оценка на val для
выбора чекпойнта, одинаковый размер сетей (2 слоя × 128, LayerNorm на входе).

  on-policy (PPO-семейство):  IPPO, MAPPO, HAPPO, MAT, PPO (один агент, факторизованное действие)
  on-policy actor-critic:     COMA (контрфактический baseline)
  off-policy:                 MADDPG (Gumbel-Softmax для дискретных действий), VDN, QMIX

Награда внутри обучения сдвинута на +1 за шаг (расписание ≈ 0): длина эпизода фиксирована (24),
поэтому сдвиг не меняет оптимальную политику, но упрощает обучение критиков.

Источники: Schulman et al. 2017 (PPO); de Witt et al. 2020 (IPPO); Yu et al. 2022 (MAPPO); Kuba et al. 2022
(HAPPO); Wen et al. 2022 (MAT); Foerster et al. 2018 (COMA); Lowe et al. 2017 (MADDPG); Sunehag et al. 2018
(VDN); Rashid et al. 2018 (QMIX). Гиперпараметры по умолчанию — из этих работ и их библиотек (MAPPO, PyMARL).
"""

from __future__ import annotations

import copy
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from marl_env import H, MetroEnv

REWARD_SHIFT = 1.0
HID = 128

DEFAULTS = {
    "common": dict(gamma=0.99, hid=HID, grad_clip=10.0),
    "ppo_family": dict(lr=5e-4, lr_critic=5e-4, n_days=32, epochs=5, minibatches=4, clip=0.2, ent=0.01,
                       gae_lambda=0.95),
    "coma": dict(lr=5e-4, lr_critic=5e-4, n_days=8, critic_epochs=2, minibatches=4, td_lambda=0.8, ent=0.01,
                 target_every=4),
    "maddpg": dict(lr=5e-4, lr_critic=5e-4, n_days=8, buffer=200_000, batch=256, upd_per_step=0.1, tau=0.01,
                   gumbel_tau=1.0, logit_reg=1e-3, warmup_steps=5_000),
    "q_family": dict(lr=5e-4, n_days=8, buffer_eps=5_000, batch_eps=32, upd_per_ep=1.0, eps_start=1.0,
                     eps_end=0.05, eps_frac=0.15, target_every_eps=200, mix_embed=32, hyper_hid=64),
}
FAMILY = {"ippo": "ppo_family", "mappo": "ppo_family", "happo": "ppo_family", "mat": "ppo_family",
          "ppo": "ppo_family", "coma": "coma", "maddpg": "maddpg", "vdn": "q_family", "qmix": "q_family"}
ALGOS = ["ippo", "mappo", "happo", "mat", "coma", "maddpg", "vdn", "qmix"]       # 8 MARL
BASELINES_LEARNED = ["ppo"]
SIMPLICITY = ["ippo", "vdn", "mappo", "ppo", "qmix", "maddpg", "coma", "happo", "mat"]  # правило 4 выбора
TITLE = {"ippo": "IPPO", "mappo": "MAPPO", "happo": "HAPPO", "mat": "MAT", "coma": "COMA", "maddpg": "MADDPG",
         "vdn": "VDN", "qmix": "QMIX", "ppo": "PPO (один агент)", "schedule": "Расписание MTA",
         "mpc_m4w": "MPC: медиана 4 нед.", "mpc_norm": "MPC: norm_level", "mpc_hyb": "MPC: гибрид ВКР",
         "mpc_oracle": "Оракул (идеальный прогноз)"}


def config(algo: str, **over) -> dict:
    c = {**DEFAULTS["common"], **DEFAULTS[FAMILY[algo]]}
    c.update(over)
    return c


# =============================================================================
# Сети
# =============================================================================
def mlp(inp, out, hid=HID, layers=2, out_gain=1.0):
    mods, d = [nn.LayerNorm(inp)], inp
    for _ in range(layers):
        lin = nn.Linear(d, hid)
        nn.init.orthogonal_(lin.weight, math.sqrt(2))
        nn.init.zeros_(lin.bias)
        mods += [lin, nn.ReLU()]
        d = hid
    last = nn.Linear(d, out)
    nn.init.orthogonal_(last.weight, out_gain)
    nn.init.zeros_(last.bias)
    return nn.Sequential(*mods, last)


class ValueNorm:
    """Нормировка целей критика скользящими средним и дисперсией (как ValueNorm в MAPPO)."""

    def __init__(self, beta=0.999):
        self.beta, self.m, self.v, self.w = beta, 0.0, 1.0, 0.0

    def update(self, x):
        x = x.detach().reshape(-1)
        b = self.beta
        self.m = b * self.m + (1 - b) * float(x.mean())
        self.v = b * self.v + (1 - b) * float((x ** 2).mean())
        self.w = b * self.w + (1 - b)

    @property
    def mean(self):
        return self.m / max(self.w, 1e-6)

    @property
    def std(self):
        return math.sqrt(max(self.v / max(self.w, 1e-6) - self.mean ** 2, 1e-6))

    def norm(self, x):
        return (x - self.mean) / self.std

    def denorm(self, x):
        return x * self.std + self.mean


def gae(rew, val, gamma, lam):
    """rew (T, …), val (T+1, …) с нулём в конце эпизода → advantages, returns."""
    T = rew.shape[0]
    adv = torch.zeros_like(rew)
    last = torch.zeros_like(rew[0])
    for t in reversed(range(T)):
        delta = rew[t] + gamma * val[t + 1] - val[t]
        last = delta + gamma * lam * last
        adv[t] = last
    return adv, adv + val[:-1]


# =============================================================================
# Базовый класс: цикл обучения с оценкой на val и выбором лучшего чекпойнта
# =============================================================================
class Learner:
    off_policy = False

    def __init__(self, env: MetroEnv, cfg: dict, seed: int):
        self.cfg, self.seed = cfg, seed
        self.N, self.od, self.sd, self.A = env.N, env.obs_dim, env.state_dim, env.n_act
        self.rng = np.random.default_rng(seed)
        torch.manual_seed(seed)
        self.steps = 0
        self.history = []

    # --- переопределяется ---
    def act(self, env, obs, state, deterministic=True):
        raise NotImplementedError

    def iterate(self, env, days_pool):
        """Один цикл: сбор данных + обновление. Возвращает (число шагов среды, средняя награда эпизода)."""
        raise NotImplementedError

    def modules(self) -> dict:
        raise NotImplementedError

    # --- общее ---
    def state_dict(self):
        return {k: copy.deepcopy(m.state_dict()) for k, m in self.modules().items()}

    def load_state_dict(self, sd):
        for k, m in self.modules().items():
            m.load_state_dict(sd[k])

    def n_params(self):
        return int(sum(p.numel() for m in self.modules().values() for p in m.parameters()))

    def train(self, env, train_days, budget, eval_fn, n_evals=12, log=None, time_limit=None):
        """budget — шагов среды (сутки × 24, по всем параллельным суткам). eval_fn(self) → оценка на val (больше — лучше)."""
        t0 = time.perf_counter()
        next_eval = budget / n_evals
        best, best_sd, best_step = -1e18, None, 0
        ep_rewards = []
        while self.steps < budget:
            n, r = self.iterate(env, train_days)
            self.steps += n
            ep_rewards.append(r)
            if self.steps >= next_eval or self.steps >= budget:
                with torch.no_grad():
                    v = float(eval_fn(self))
                rec = {"step": self.steps, "train_reward": float(np.mean(ep_rewards[-20:])), "val_score": v,
                       "sec": time.perf_counter() - t0}
                self.history.append(rec)
                if v > best:
                    best, best_sd, best_step = v, self.state_dict(), self.steps
                if log:
                    log(f"    {self.steps:>8d} шагов  train {rec['train_reward']:+.3f}  val {v:+.4f}  "
                        f"{rec['sec']:.0f} c")
                next_eval += budget / n_evals
            if time_limit and time.perf_counter() - t0 > time_limit:
                if log:
                    log("    остановлено по лимиту времени")
                break
        if best_sd is not None:
            self.load_state_dict(best_sd)
        self.best_step, self.best_val = best_step, best
        self.train_sec = time.perf_counter() - t0
        return self.history

    def sample_days(self, pool, n):
        return self.rng.choice(pool, size=n, replace=True)

    @torch.no_grad()
    def collect(self, env, days, policy_fn):
        """Прогон эпизодов с записью траекторий (T = 24). policy_fn(obs, state) → actions, extra(dict тензоров)."""
        obs, state = env.reset(days)
        buf = {"obs": [], "state": [], "act": [], "rew": [], "extra": []}
        done = False
        while not done:
            a, ex = policy_fn(env, obs, state)
            buf["obs"].append(obs)
            buf["state"].append(state)
            buf["act"].append(a)
            buf["extra"].append(ex)
            obs, state, r, done, info = env.step(a)
            buf["rew"].append(r + REWARD_SHIFT)
        out = {k: torch.stack(buf[k]) for k in ["obs", "state", "act", "rew"]}
        for k in buf["extra"][0]:
            out[k] = torch.stack([e[k] for e in buf["extra"]])
        out["ep_reward"] = float(sum(buf["rew"]).mean()) - REWARD_SHIFT * H
        return out


def categorical(logits, deterministic):
    if deterministic:
        return logits.argmax(-1)
    return torch.distributions.Categorical(logits=logits).sample()


def clip_step(opt, params, max_norm):
    nn.utils.clip_grad_norm_(params, max_norm)
    opt.step()


# =============================================================================
# PPO-семейство
# =============================================================================
class PPOBase(Learner):
    """Общий код для IPPO / MAPPO / HAPPO / PPO (один агент)."""
    centralized_v = True

    def __init__(self, env, cfg, seed):
        super().__init__(env, cfg, seed)
        self.vnorm = ValueNorm()
        self.build()

    def build(self):
        raise NotImplementedError

    def logits(self, obs, state):                # → (B, N, A)
        raise NotImplementedError

    def value(self, obs, state):                 # → (B,) или (B, N)
        raise NotImplementedError

    def act(self, env, obs, state, deterministic=True):
        return categorical(self.logits(obs, state), deterministic)

    def iterate(self, env, pool):
        c = self.cfg
        days = self.sample_days(pool, c["n_days"])

        def pf(env_, o, s):
            lg = self.logits(o, s)
            a = categorical(lg, False)
            lp = torch.log_softmax(lg, -1).gather(-1, a[..., None])[..., 0]
            return a, {"logp": lp, "v": self.value(o, s)}

        tr = self.collect(env, days, pf)
        self.update(tr)
        return c["n_days"] * H, tr["ep_reward"]

    def advantages(self, tr):
        c = self.cfg
        v = self.vnorm.denorm(tr["v"])                                 # (T, B) или (T, B, N)
        v = torch.cat([v, torch.zeros_like(v[:1])], 0)
        rew = tr["rew"] if v.dim() == 2 else tr["rew"][..., None].expand_as(v[:-1])
        adv, ret = gae(rew, v, c["gamma"], c["gae_lambda"])
        self.vnorm.update(ret)
        return adv, ret

    def policy_loss(self, lg, act, old_lp, adv, w=None):
        c = self.cfg
        lp_all = torch.log_softmax(lg, -1)
        lp = lp_all.gather(-1, act[..., None])[..., 0]
        ratio = torch.exp(lp - old_lp)
        if w is not None:
            adv = adv * w
        s1, s2 = ratio * adv, ratio.clamp(1 - c["clip"], 1 + c["clip"]) * adv
        ent = -(lp_all.exp() * lp_all).sum(-1)
        return -torch.min(s1, s2).mean() - c["ent"] * ent.mean()

    def update(self, tr):
        c = self.cfg
        adv, ret = self.advantages(tr)
        T, B = tr["rew"].shape
        flat = lambda x: x.reshape(T * B, *x.shape[2:])
        obs, st, act, olp = flat(tr["obs"]), flat(tr["state"]), flat(tr["act"]), flat(tr["logp"])
        adv, ret = flat(adv), flat(ret)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        if adv.dim() == 1:
            adv = adv[:, None].expand(-1, self.N)
        tgt = self.vnorm.norm(ret)
        n = T * B
        for _ in range(c["epochs"]):
            perm = torch.randperm(n)
            for mb in perm.chunk(c["minibatches"]):
                lg = self.logits(obs[mb], st[mb])
                loss_pi = self.policy_loss(lg, act[mb], olp[mb], adv[mb])
                self.opt_pi.zero_grad()
                loss_pi.backward()
                clip_step(self.opt_pi, self.pi_params(), c["grad_clip"])
                v = self.value(obs[mb], st[mb])
                loss_v = F.mse_loss(v, tgt[mb])
                self.opt_v.zero_grad()
                loss_v.backward()
                clip_step(self.opt_v, self.v_params(), c["grad_clip"])

    def pi_params(self):
        return list(self.actor.parameters())

    def v_params(self):
        return list(self.critic.parameters())

    def modules(self):
        return {"actor": self.actor, "critic": self.critic}


class IPPO(PPOBase):
    """Независимые PPO-агенты с общими параметрами; критик видит только своё наблюдение."""

    def build(self):
        c = self.cfg
        self.actor = mlp(self.od, self.A, c["hid"], out_gain=0.01)
        self.critic = mlp(self.od, 1, c["hid"])
        self.opt_pi = torch.optim.Adam(self.actor.parameters(), lr=c["lr"], eps=1e-5)
        self.opt_v = torch.optim.Adam(self.critic.parameters(), lr=c["lr_critic"], eps=1e-5)

    def logits(self, obs, state):
        return self.actor(obs)

    def value(self, obs, state):
        return self.critic(obs)[..., 0]


class MAPPO(IPPO):
    """PPO с централизованным критиком V(s) (CTDE), общий актор по наблюдениям агентов."""

    def build(self):
        c = self.cfg
        self.actor = mlp(self.od, self.A, c["hid"], out_gain=0.01)
        self.critic = mlp(self.sd, 1, c["hid"])
        self.opt_pi = torch.optim.Adam(self.actor.parameters(), lr=c["lr"], eps=1e-5)
        self.opt_v = torch.optim.Adam(self.critic.parameters(), lr=c["lr_critic"], eps=1e-5)

    def value(self, obs, state):
        return self.critic(state)[..., 0]


class SinglePPO(MAPPO):
    """Контрольный baseline: один агент PPO управляет всеми группами сразу (вход — полное состояние,
    действие факторизовано по агентам; отношение вероятностей — по совместному действию)."""

    def build(self):
        c = self.cfg
        self.actor = mlp(self.sd, self.N * self.A, c["hid"], out_gain=0.01)
        self.critic = mlp(self.sd, 1, c["hid"])
        self.opt_pi = torch.optim.Adam(self.actor.parameters(), lr=c["lr"], eps=1e-5)
        self.opt_v = torch.optim.Adam(self.critic.parameters(), lr=c["lr_critic"], eps=1e-5)

    def logits(self, obs, state):
        return self.actor(state).reshape(*state.shape[:-1], self.N, self.A)

    def policy_loss(self, lg, act, old_lp, adv, w=None):
        c = self.cfg
        lp_all = torch.log_softmax(lg, -1)
        lp = lp_all.gather(-1, act[..., None])[..., 0].sum(-1)
        ratio = torch.exp(lp - old_lp.sum(-1))
        a = adv[:, 0]
        s1, s2 = ratio * a, ratio.clamp(1 - c["clip"], 1 + c["clip"]) * a
        ent = -(lp_all.exp() * lp_all).sum(-1).sum(-1)
        return -torch.min(s1, s2).mean() - c["ent"] * ent.mean()


class HAPPO(MAPPO):
    """Heterogeneous-Agent PPO: у каждого агента своя сеть; агенты обновляются по очереди в случайном порядке,
    преимущество взвешивается произведением отношений вероятностей уже обновлённых агентов (Kuba et al., 2022)."""

    def build(self):
        c = self.cfg
        self.actors = nn.ModuleList([mlp(self.od, self.A, c["hid"], out_gain=0.01) for _ in range(self.N)])
        self.critic = mlp(self.sd, 1, c["hid"])
        self.opts = [torch.optim.Adam(a.parameters(), lr=c["lr"], eps=1e-5) for a in self.actors]
        self.opt_v = torch.optim.Adam(self.critic.parameters(), lr=c["lr_critic"], eps=1e-5)

    def logits(self, obs, state):
        return torch.stack([self.actors[i](obs[..., i, :]) for i in range(self.N)], -2)

    def modules(self):
        return {"actors": self.actors, "critic": self.critic}

    def update(self, tr):
        c = self.cfg
        adv, ret = self.advantages(tr)
        T, B = tr["rew"].shape
        flat = lambda x: x.reshape(T * B, *x.shape[2:])
        obs, st, act, olp = flat(tr["obs"]), flat(tr["state"]), flat(tr["act"]), flat(tr["logp"])
        adv, ret = flat(adv), flat(ret)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        tgt = self.vnorm.norm(ret)
        n = T * B
        for _ in range(c["epochs"]):                                   # критик
            for mb in torch.randperm(n).chunk(c["minibatches"]):
                loss_v = F.mse_loss(self.value(obs[mb], st[mb]), tgt[mb])
                self.opt_v.zero_grad()
                loss_v.backward()
                clip_step(self.opt_v, self.v_params(), c["grad_clip"])
        M = torch.ones(n)
        for i in torch.randperm(self.N).tolist():                      # акторы по очереди
            net, opt = self.actors[i], self.opts[i]
            for _ in range(c["epochs"]):
                for mb in torch.randperm(n).chunk(c["minibatches"]):
                    lg = net(obs[mb, i])
                    loss = self.policy_loss(lg, act[mb, i], olp[mb, i], adv[mb], w=M[mb])
                    opt.zero_grad()
                    loss.backward()
                    clip_step(opt, net.parameters(), c["grad_clip"])
            with torch.no_grad():
                new_lp = torch.log_softmax(net(obs[:, i]), -1).gather(-1, act[:, i, None])[:, 0]
                M = M * torch.exp(new_lp - olp[:, i]).clamp(0.0, 10.0)


# =============================================================================
# MAT — Multi-Agent Transformer (Wen et al., 2022)
# =============================================================================
class MATNet(nn.Module):
    def __init__(self, od, N, A, d=64, heads=2):
        super().__init__()
        self.N, self.A, self.d = N, A, d
        self.obs_emb = nn.Sequential(nn.LayerNorm(od), nn.Linear(od, d), nn.GELU())
        self.enc = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.v_head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.act_emb = nn.Sequential(nn.Linear(A + 1, d), nn.GELU())
        self.pos = nn.Parameter(torch.zeros(N, d))
        self.dec = nn.TransformerDecoderLayer(d, heads, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.pi_head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, A))
        nn.init.normal_(self.pos, std=0.02)
        self.register_buffer("mask", torch.triu(torch.full((N, N), float("-inf")), 1))

    def encode(self, obs):
        rep = self.enc(self.obs_emb(obs))
        return rep, self.v_head(rep)[..., 0]

    def tokens(self, act_prev):
        """act_prev (…, N) действия предыдущих агентов (сдвиг на 1); первый токен — «старт»."""
        B = act_prev.shape[:-1]
        x = torch.zeros(*B, self.N, self.A + 1)
        x[..., 0, 0] = 1.0
        oh = F.one_hot(act_prev[..., :-1], self.A).float()
        x[..., 1:, 1:] = oh
        return x

    def decode(self, tok, rep, upto=None):
        n = tok.shape[-2] if upto is None else upto
        h = self.act_emb(tok[..., :n, :]) + self.pos[:n]
        h = self.dec(h, rep, tgt_mask=self.mask[:n, :n])
        return self.pi_head(h)


class MAT(PPOBase):
    def build(self):
        c = self.cfg
        self.net = MATNet(self.od, self.N, self.A)
        self.opt_pi = torch.optim.Adam(self.net.parameters(), lr=c["lr"], eps=1e-5)

    def modules(self):
        return {"net": self.net}

    def value(self, obs, state):
        return self.net.encode(obs)[1]

    @torch.no_grad()
    def _sample(self, obs, deterministic):
        rep, v = self.net.encode(obs)
        B = obs.shape[0]
        act = torch.zeros(B, self.N, dtype=torch.long)
        logits = torch.zeros(B, self.N, self.A)
        tok = torch.zeros(B, self.N, self.A + 1)
        tok[:, 0, 0] = 1.0
        for i in range(self.N):
            lg = self.net.decode(tok, rep, upto=i + 1)[:, i]
            a = categorical(lg, deterministic)
            act[:, i], logits[:, i] = a, lg
            if i + 1 < self.N:
                tok[:, i + 1, 1:] = F.one_hot(a, self.A).float()
        return act, logits, v

    def act(self, env, obs, state, deterministic=True):
        return self._sample(obs, deterministic)[0]

    def iterate(self, env, pool):
        c = self.cfg
        days = self.sample_days(pool, c["n_days"])

        def pf(env_, o, s):
            a, lg, v = self._sample(o, False)
            lp = torch.log_softmax(lg, -1).gather(-1, a[..., None])[..., 0]
            return a, {"logp": lp, "v": v}

        tr = self.collect(env, days, pf)
        self.update(tr)
        return c["n_days"] * H, tr["ep_reward"]

    def update(self, tr):
        c = self.cfg
        adv, ret = self.advantages(tr)
        T, B = tr["rew"].shape
        flat = lambda x: x.reshape(T * B, *x.shape[2:])
        obs, act, olp, adv, ret = flat(tr["obs"]), flat(tr["act"]), flat(tr["logp"]), flat(adv), flat(ret)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        tgt = self.vnorm.norm(ret)
        n = T * B
        for _ in range(c["epochs"]):
            for mb in torch.randperm(n).chunk(c["minibatches"]):
                rep, v = self.net.encode(obs[mb])
                lg = self.net.decode(self.net.tokens(act[mb]), rep)
                loss = self.policy_loss(lg, act[mb], olp[mb], adv[mb]) + F.mse_loss(v, tgt[mb])
                self.opt_pi.zero_grad()
                loss.backward()
                clip_step(self.opt_pi, self.net.parameters(), c["grad_clip"])


# =============================================================================
# COMA — контрфактический многоагентный градиент политики (Foerster et al., 2018)
# =============================================================================
class COMA(Learner):
    def __init__(self, env, cfg, seed):
        super().__init__(env, cfg, seed)
        c = cfg
        self.actor = mlp(self.od, self.A, c["hid"], out_gain=0.01)
        cin = self.sd + self.od + self.N * self.A
        self.critic = mlp(cin, self.A, c["hid"])
        self.critic_t = copy.deepcopy(self.critic)
        self.opt_pi = torch.optim.Adam(self.actor.parameters(), lr=c["lr"], eps=1e-5)
        self.opt_q = torch.optim.Adam(self.critic.parameters(), lr=c["lr_critic"], eps=1e-5)
        self.it = 0

    def modules(self):
        return {"actor": self.actor, "critic": self.critic}

    def act(self, env, obs, state, deterministic=True):
        return categorical(self.actor(obs), deterministic)

    def critic_in(self, state, obs, act):
        """Вход критика агента i: состояние, наблюдение i, действия остальных (действие i обнулено)."""
        N, A = self.N, self.A
        oh = F.one_hot(act, A).float()                                   # (…, N, A)
        others = oh[..., None, :, :].expand(*oh.shape[:-2], N, N, A).clone()
        idx = torch.arange(N)
        others[..., idx, idx, :] = 0.0
        others = others.reshape(*oh.shape[:-2], N, N * A)
        st = state[..., None, :].expand(*state.shape[:-1], N, state.shape[-1])
        return torch.cat([st, obs, others], -1)

    def iterate(self, env, pool):
        c = self.cfg
        days = self.sample_days(pool, c["n_days"])

        def pf(env_, o, s):
            return categorical(self.actor(o), False), {}

        tr = self.collect(env, days, pf)
        T, B = tr["rew"].shape
        x = self.critic_in(tr["state"], tr["obs"], tr["act"])            # (T, B, N, cin)
        with torch.no_grad():
            q_t = self.critic_t(x).gather(-1, tr["act"][..., None])[..., 0]   # (T, B, N)
            G = torch.zeros_like(q_t)
            nxt = torch.zeros_like(q_t[0])
            for t in reversed(range(T)):
                boot = (1 - c["td_lambda"]) * q_t[t + 1] if t + 1 < T else torch.zeros_like(nxt)
                G[t] = tr["rew"][t][:, None] + c["gamma"] * (boot + c["td_lambda"] * nxt)
                nxt = G[t] if t + 1 <= T else nxt
        xf, af, Gf = x.reshape(T * B, self.N, -1), tr["act"].reshape(T * B, self.N), G.reshape(T * B, self.N)
        for _ in range(c["critic_epochs"]):
            for mb in torch.randperm(T * B).chunk(c["minibatches"]):
                q = self.critic(xf[mb]).gather(-1, af[mb][..., None])[..., 0]
                loss = F.mse_loss(q, Gf[mb])
                self.opt_q.zero_grad()
                loss.backward()
                clip_step(self.opt_q, self.critic.parameters(), c["grad_clip"])
        with torch.no_grad():
            q_all = self.critic(xf)                                      # (TB, N, A)
        lg = self.actor(tr["obs"].reshape(T * B, self.N, -1))
        lp_all = torch.log_softmax(lg, -1)
        pi = lp_all.exp()
        base = (pi.detach() * q_all).sum(-1)
        qa = q_all.gather(-1, af[..., None])[..., 0]
        adv = qa - base
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        lp = lp_all.gather(-1, af[..., None])[..., 0]
        ent = -(pi * lp_all).sum(-1)
        loss = -(adv.detach() * lp).mean() - c["ent"] * ent.mean()
        self.opt_pi.zero_grad()
        loss.backward()
        clip_step(self.opt_pi, self.actor.parameters(), c["grad_clip"])
        self.it += 1
        if self.it % c["target_every"] == 0:
            self.critic_t.load_state_dict(self.critic.state_dict())
        return c["n_days"] * H, tr["ep_reward"]


# =============================================================================
# Буфер переходов для off-policy
# =============================================================================
class Replay:
    def __init__(self, cap):
        self.cap, self.n, self.i, self.d = cap, 0, 0, None

    def add(self, **kw):
        k0 = next(iter(kw.values()))
        m = k0.shape[0]
        if self.d is None:
            self.d = {k: torch.zeros((self.cap,) + v.shape[1:], dtype=v.dtype) for k, v in kw.items()}
        idx = (torch.arange(m) + self.i) % self.cap
        for k, v in kw.items():
            self.d[k][idx] = v
        self.i = (self.i + m) % self.cap
        self.n = min(self.n + m, self.cap)

    def sample(self, m, rng):
        j = torch.as_tensor(rng.integers(0, self.n, size=m))
        return {k: v[j] for k, v in self.d.items()}


def episodes_to_transitions(tr):
    """(T, B, …) → переходы (T·B, …) с next и done; на последнем шаге next — нули, done = 1."""
    T, B = tr["rew"].shape
    nxt = lambda x: torch.cat([x[1:], torch.zeros_like(x[:1])], 0)
    done = torch.zeros(T, B)
    done[-1] = 1.0
    f = lambda x: x.reshape(T * B, *x.shape[2:])
    return dict(obs=f(tr["obs"]), state=f(tr["state"]), act=f(tr["act"]), rew=f(tr["rew"]),
                obs2=f(nxt(tr["obs"])), state2=f(nxt(tr["state"])), done=f(done))


# =============================================================================
# MADDPG (дискретные действия через Gumbel-Softmax)
# =============================================================================
class MADDPG(Learner):
    off_policy = True

    def __init__(self, env, cfg, seed):
        super().__init__(env, cfg, seed)
        c = cfg
        self.actor = mlp(self.od, self.A, c["hid"], out_gain=0.01)
        self.critic = mlp(self.sd + self.N * self.A, 1, c["hid"])
        self.actor_t, self.critic_t = copy.deepcopy(self.actor), copy.deepcopy(self.critic)
        self.opt_pi = torch.optim.Adam(self.actor.parameters(), lr=c["lr"])
        self.opt_q = torch.optim.Adam(self.critic.parameters(), lr=c["lr_critic"])
        self.buf = Replay(c["buffer"])

    def modules(self):
        return {"actor": self.actor, "critic": self.critic}

    def act(self, env, obs, state, deterministic=True):
        lg = self.actor(obs)
        if deterministic:
            return lg.argmax(-1)
        return F.gumbel_softmax(lg, tau=self.cfg["gumbel_tau"], hard=True).argmax(-1)

    def q(self, net, state, oh):
        return net(torch.cat([state, oh.reshape(*oh.shape[:-2], -1)], -1))[..., 0]

    def iterate(self, env, pool):
        c = self.cfg
        days = self.sample_days(pool, c["n_days"])
        warm = self.steps < c["warmup_steps"]

        def pf(env_, o, s):
            if warm:
                return torch.randint(0, self.A, o.shape[:-1]), {}
            return self.act(env_, o, s, deterministic=False), {}

        tr = self.collect(env, days, pf)
        self.buf.add(**episodes_to_transitions(tr))
        n = c["n_days"] * H
        if self.buf.n >= c["batch"]:
            for _ in range(max(1, int(n * c["upd_per_step"]))):
                self.update(self.buf.sample(c["batch"], self.rng))
        return n, tr["ep_reward"]

    def update(self, b):
        c = self.cfg
        oh = F.one_hot(b["act"], self.A).float()
        with torch.no_grad():
            a2 = F.one_hot(self.actor_t(b["obs2"]).argmax(-1), self.A).float()
            y = b["rew"] + c["gamma"] * (1 - b["done"]) * self.q(self.critic_t, b["state2"], a2)
        loss_q = F.mse_loss(self.q(self.critic, b["state"], oh), y)
        self.opt_q.zero_grad()
        loss_q.backward()
        clip_step(self.opt_q, self.critic.parameters(), c["grad_clip"])
        lg = self.actor(b["obs"])                                          # (m, N, A)
        g = F.gumbel_softmax(lg, tau=c["gumbel_tau"], hard=True)
        loss_pi = 0.0
        for i in range(self.N):                                            # подстановка действия агента i
            joint = torch.cat([oh[:, :i], g[:, i:i + 1], oh[:, i + 1:]], 1)
            loss_pi = loss_pi - self.q(self.critic, b["state"], joint).mean()
        loss_pi = loss_pi / self.N + c["logit_reg"] * (lg ** 2).mean()
        self.opt_pi.zero_grad()
        loss_pi.backward()
        clip_step(self.opt_pi, self.actor.parameters(), c["grad_clip"])
        with torch.no_grad():
            for net, tgt in [(self.actor, self.actor_t), (self.critic, self.critic_t)]:
                for p, pt in zip(net.parameters(), tgt.parameters()):
                    pt.mul_(1 - c["tau"]).add_(c["tau"] * p)


# =============================================================================
# VDN и QMIX
# =============================================================================
class QMixer(nn.Module):
    def __init__(self, N, sd, embed=32, hyper=64):
        super().__init__()
        self.N, self.e = N, embed
        self.ln = nn.LayerNorm(sd)
        self.w1 = nn.Sequential(nn.Linear(sd, hyper), nn.ReLU(), nn.Linear(hyper, N * embed))
        self.b1 = nn.Linear(sd, embed)
        self.w2 = nn.Sequential(nn.Linear(sd, hyper), nn.ReLU(), nn.Linear(hyper, embed))
        self.b2 = nn.Sequential(nn.Linear(sd, embed), nn.ReLU(), nn.Linear(embed, 1))

    def forward(self, q, state):                     # q (m, N), state (m, sd) → (m,)
        s = self.ln(state)
        w1 = self.w1(s).abs().view(-1, self.N, self.e)
        h = F.elu(torch.bmm(q[:, None, :], w1)[:, 0] + self.b1(s))
        w2 = self.w2(s).abs()[..., None]
        return (torch.bmm(h[:, None, :], w2)[:, 0, 0] + self.b2(s)[:, 0])


class VDN(Learner):
    off_policy = True
    mixer_cls = None

    def __init__(self, env, cfg, seed):
        super().__init__(env, cfg, seed)
        c = cfg
        self.agent = mlp(self.od, self.A, c["hid"])
        self.mixer = QMixer(self.N, self.sd, c["mix_embed"], c["hyper_hid"]) if self.mixer_cls == "qmix" else None
        self.agent_t = copy.deepcopy(self.agent)
        self.mixer_t = copy.deepcopy(self.mixer) if self.mixer is not None else None
        params = list(self.agent.parameters()) + (list(self.mixer.parameters()) if self.mixer is not None else [])
        self.params = params
        self.opt = torch.optim.Adam(params, lr=c["lr"])
        self.buf = Replay(c["buffer_eps"] * H)
        self.eps_count = 0
        self.budget = None

    def modules(self):
        d = {"agent": self.agent}
        if self.mixer is not None:
            d["mixer"] = self.mixer
        return d

    def eps(self):
        c = self.cfg
        frac = min(1.0, self.steps / max(1.0, c["eps_frac"] * (self.budget or 1e6)))
        return c["eps_start"] + frac * (c["eps_end"] - c["eps_start"])

    def act(self, env, obs, state, deterministic=True):
        q = self.agent(obs)
        if deterministic:
            return q.argmax(-1)
        a = q.argmax(-1)
        rnd = torch.rand(a.shape) < self.eps()
        return torch.where(rnd, torch.randint(0, self.A, a.shape), a)

    def mix(self, mixer, q, state):
        return q.sum(-1) if mixer is None else mixer(q, state)

    def train(self, env, train_days, budget, eval_fn, **kw):
        self.budget = budget
        return super().train(env, train_days, budget, eval_fn, **kw)

    def iterate(self, env, pool):
        c = self.cfg
        days = self.sample_days(pool, c["n_days"])
        tr = self.collect(env, days, lambda e, o, s: (self.act(e, o, s, deterministic=False), {}))
        self.buf.add(**episodes_to_transitions(tr))
        prev = self.eps_count
        self.eps_count += c["n_days"]
        if self.buf.n >= c["batch_eps"] * H:
            for _ in range(max(1, int(c["n_days"] * c["upd_per_ep"]))):
                self.update(self.buf.sample(c["batch_eps"] * H, self.rng))
        if self.eps_count // c["target_every_eps"] > prev // c["target_every_eps"]:
            self.agent_t.load_state_dict(self.agent.state_dict())
            if self.mixer is not None:
                self.mixer_t.load_state_dict(self.mixer.state_dict())
        return c["n_days"] * H, tr["ep_reward"]

    def update(self, b):
        c = self.cfg
        q = self.agent(b["obs"]).gather(-1, b["act"][..., None])[..., 0]          # (m, N)
        qt = self.mix(self.mixer, q, b["state"])
        with torch.no_grad():
            a2 = self.agent(b["obs2"]).argmax(-1, keepdim=True)                     # double Q
            q2 = self.agent_t(b["obs2"]).gather(-1, a2)[..., 0]
            y = b["rew"] + c["gamma"] * (1 - b["done"]) * self.mix(self.mixer_t, q2, b["state2"])
        loss = F.smooth_l1_loss(qt, y)
        self.opt.zero_grad()
        loss.backward()
        clip_step(self.opt, self.params, c["grad_clip"])


class QMIX(VDN):
    mixer_cls = "qmix"


CLASSES = {"ippo": IPPO, "mappo": MAPPO, "happo": HAPPO, "mat": MAT, "ppo": SinglePPO, "coma": COMA,
           "maddpg": MADDPG, "vdn": VDN, "qmix": QMIX}


def make(algo: str, env: MetroEnv, seed: int, **over) -> Learner:
    torch.manual_seed(seed)
    np.random.seed(seed)
    return CLASSES[algo](env, config(algo, **over), seed)

In [ ]:
%%writefile /kaggle/working/marl_eval.py
"""
marl_eval — оценка политик ЛР3: суточные метрики, доверительные интервалы, сводный балл и правило выбора,
кооперация агентов, стресс-тесты, запуск обучения (в т.ч. параллельно) с журналом параметров.

Соответствие метрик заданию (финансовый пример → метро), PROTOCOL.md разд. 6:
  доходность            → выигрыш: 1 − потери политики / потери расписания за сутки, среднее по суткам
  Sharpe                → IR: среднее / ст. откл. суточного выигрыша
  Sortino               → среднее / нижнее отклонение (только сутки хуже расписания)
  максимальная просадка → худшие сутки (минимальный суточный выигрыш)
  CVaR 95%              → средний выигрыш в 5% худших суток
  оборот                → перераспределение поездо-часов между часами, % суточного бюджета
  издержки              → поездо-часы (% бюджета) и штраф за смену частоты
  нарушения лимитов     → перерасход бюджета > 0,5% (доля суток); часы с наполнением выше норматива;
                          вмешательства защитного слоя
"""

from __future__ import annotations

import json
import os
import time
import traceback
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from marl_env import (A_SCHED, GROUPS, H, EnvData, EnvParams, MetroEnv, MPCPolicy, SchedulePolicy, rollout)

TOL_OVERRUN = 0.005
WEIGHTS = {"gain": 0.35, "ir": 0.20, "cvar": 0.20, "crowded": 0.10, "turnover": 0.10, "seed_sd": 0.05}
HIGHER_BETTER = {"gain": True, "ir": True, "cvar": True, "crowded": False, "turnover": False, "seed_sd": False}
BOOT = 2000


def to_md(df: pd.DataFrame, floatfmt="{:.4g}") -> str:
    """Markdown-таблица без зависимости от tabulate."""
    cols = [str(c) for c in df.columns]
    fmt = lambda v: floatfmt.format(v) if isinstance(v, (float, np.floating)) and np.isfinite(v) else str(v)
    rows = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(rows)


# =============================================================================
# Суточные метрики
# =============================================================================
def daily_frame(r: dict, env: MetroEnv, policy: str, seed, part: str) -> pd.DataFrame:
    b = r["day"]
    bud = env.budget[torch.as_tensor(b)].numpy()
    G = env.G
    df = pd.DataFrame({
        "policy": policy, "seed": seed, "part": part, "day": env.data.day[b],
        "J": r["J"], "J_sched": r["J_sched"], "wait": r["wait"], "extra": r["extra"], "crowd": r["crowd"],
        "pax": r["pax"], "pax_crowded": r["pax_crowded"], "breach_share": r["breach"] / (H * G),
        "denied": r["denied"], "used_frac": r["used_frac"], "overrun": r["overrun"],
        "turnover": r["switch"] / bud, "clip_share": r["clip"] / (H * G), "cap_share": r["capcut"] / H,
        "reward": r["reward"],
    })
    df["gain"] = 1.0 - df["J"] / df["J_sched"]
    for k, v in env.data.flags.items():
        df[f"flag_{k}"] = v[b]
    return df


def evaluate(policy, env: MetroEnv, days, name: str, seed=None, part="") -> pd.DataFrame:
    if len(days) == 0:
        return pd.DataFrame()
    t = time.perf_counter()
    r = rollout(env, policy, days)
    df = daily_frame(r, env, name, seed, part)
    df["infer_ms"] = (time.perf_counter() - t) / (len(days) * H) * 1000
    return df


def summarize(df: pd.DataFrame) -> dict:
    """Метрики по набору суток одной политики (одного seed или нескольких — тогда среднее по seed по каждым суткам)."""
    if df["seed"].nunique() > 1:
        num = df.groupby("day").mean(numeric_only=True)
    else:
        num = df.set_index("day").select_dtypes("number")
    g = num["gain"].to_numpy()
    k = max(1, int(np.ceil(0.05 * len(g))))
    down = np.sqrt(np.mean(np.minimum(g, 0) ** 2))
    sortino = g.mean() / down if down > 1e-9 else (np.inf if g.mean() > 0 else np.nan)   # ∞ — нет суток хуже расписания
    return {
        "gain": g.mean(), "ir": g.mean() / (g.std() + 1e-9), "sortino": sortino, "worst": g.min(),
        "cvar": np.sort(g)[:k].mean(), "cost_per_pax": num["J"].sum() / num["pax"].sum(),
        "wait_per_pax": num["wait"].sum() / num["pax"].sum(),
        "crowded": num["pax_crowded"].sum() / num["pax"].sum(), "breach": num["breach_share"].mean(),
        "turnover": num["turnover"].mean(), "used_frac": num["used_frac"].mean(),
        "viol_budget": (num["overrun"] > TOL_OVERRUN).mean(), "clip_share": num["clip_share"].mean(),
        "cap_share": num["cap_share"].mean(), "reward": num["reward"].mean(), "days": len(g),
    }


def table(daily: pd.DataFrame, by=("policy",)) -> pd.DataFrame:
    """Сводная таблица: метрики по среднему seed + разброс по seed (ст. откл. выигрыша и IR между seed)."""
    rows = []
    for key, grp in daily.groupby(list(by)):
        key = key if isinstance(key, tuple) else (key,)
        row = dict(zip(by, key))
        row.update(summarize(grp))
        per_seed = [summarize(s) for _, s in grp.groupby("seed", dropna=False)]
        row["n_seeds"] = len(per_seed)
        row["seed_sd"] = float(np.std([p["gain"] for p in per_seed])) if len(per_seed) > 1 else 0.0
        row["ir_seed_sd"] = float(np.std([p["ir"] for p in per_seed])) if len(per_seed) > 1 else 0.0
        if "infer_ms" in grp:
            row["infer_ms"] = grp["infer_ms"].mean()
        rows.append(row)
    return pd.DataFrame(rows)


# =============================================================================
# Доверительные интервалы: стратифицированный бутстреп (seed × недельные блоки суток)
# =============================================================================
def _week_blocks(days):
    d = pd.DatetimeIndex(days)
    wk = ((d - d.min()).days // 7).to_numpy()
    return [np.where(wk == w)[0] for w in np.unique(wk)]


def boot_ci(daily: pd.DataFrame, col="gain", n=BOOT, seed=0, alpha=0.05) -> tuple[float, float]:
    """95% ДИ среднего col: случайно выбираем seed (с возвращением) и недели (с возвращением)."""
    rng = np.random.default_rng(seed)
    piv = daily.pivot_table(index="day", columns="seed", values=col, dropna=False)
    M = piv.to_numpy()
    blocks = _week_blocks(piv.index)
    S = M.shape[1]
    est = np.empty(n)
    for i in range(n):
        ss = rng.integers(0, S, S)
        bb = np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))])
        est[i] = np.nanmean(M[np.ix_(bb, ss)])
    return float(np.quantile(est, alpha / 2)), float(np.quantile(est, 1 - alpha / 2))


def paired_diff(da: pd.DataFrame, db: pd.DataFrame, col="gain", n=BOOT, seed=0) -> dict:
    """Разница A − B (средние по seed по каждым суткам), блочный бутстреп по неделям: ДИ и двусторонний p."""
    rng = np.random.default_rng(seed)
    a = da.groupby("day")[col].mean()
    b = db.groupby("day")[col].mean()
    d = (a - b).dropna()
    blocks = _week_blocks(d.index)
    v = d.to_numpy()
    est = np.array([v[np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))])].mean()
                    for _ in range(n)])
    p = 2 * min((est <= 0).mean(), (est >= 0).mean())
    return {"diff": float(v.mean()), "lo": float(np.quantile(est, 0.025)), "hi": float(np.quantile(est, 0.975)),
            "p": float(min(1.0, p)), "days": len(v)}


# =============================================================================
# Сводный балл и правило выбора (PROTOCOL.md разд. 7)
# =============================================================================
def composite(tab: pd.DataFrame) -> pd.Series:
    sc = pd.Series(0.0, index=tab.index)
    for k, w in WEIGHTS.items():
        x = tab[k].astype(float)
        lo, hi = x.min(), x.max()
        z = (x - lo) / (hi - lo) if hi > lo else pd.Series(1.0, index=x.index)
        sc += w * (z if HIGHER_BETTER[k] else 1 - z)
    return sc


def select(tab_val: pd.DataFrame, daily_val: pd.DataFrame, simplicity: list, log=print) -> dict:
    """1) допустимы: перерасход бюджета ≤ 2% суток и нижняя граница ДИ выигрыша > 0;
    2) максимум сводного балла; 3) если более простой кандидат не хуже по выигрышу значимо (ДИ разницы
    содержит 0) и его балл отстаёт не больше чем на 0,10 — берётся более простой."""
    t = tab_val.copy()
    t["score"] = composite(t)
    t["ci_lo"] = [boot_ci(daily_val[daily_val["policy"] == p])[0] for p in t["policy"]]
    t["admissible"] = (t["viol_budget"] <= 0.02) & (t["ci_lo"] > 0)
    pool = t[t["admissible"]] if t["admissible"].any() else t
    if not t["admissible"].any():
        log("ВНИМАНИЕ: ни один алгоритм не прошёл условия допуска — выбор среди всех (записать в отступления)")
    top = pool.sort_values("score", ascending=False).iloc[0]
    choice, reason = top["policy"], "максимум сводного балла"
    rank = {p: i for i, p in enumerate(simplicity)}
    for _, row in pool.sort_values("score", ascending=False).iterrows():
        if rank.get(row["policy"], 99) < rank.get(choice, 99) and row["score"] >= top["score"] - 0.10:
            pd_ = paired_diff(daily_val[daily_val["policy"] == top["policy"]],
                              daily_val[daily_val["policy"] == row["policy"]])
            if pd_["lo"] <= 0 <= pd_["hi"]:
                choice = row["policy"]
                reason = (f"более простой, разница выигрыша с {top['policy']} незначима "
                          f"({pd_['diff'] * 100:+.2f} п.п., ДИ [{pd_['lo'] * 100:+.2f}; {pd_['hi'] * 100:+.2f}])")
    return {"choice": choice, "reason": reason, "table": t.sort_values("score", ascending=False)}


# =============================================================================
# Кооперация агентов
# =============================================================================
def cooperation(policy, env: MetroEnv, days, name="") -> dict:
    """Вклад агентов и согласованность действий на сутках days (детерминированная политика).

    loo_i   — насколько падает выигрыш, если действие агента i заменить расписанием (остальные — политика);
    solo_i  — выигрыш, если по политике действует только агент i (остальные — расписание);
    synergy — выигрыш всех минус сумма solo: > 0 — агенты дополняют друг друга (перераспределение ресурса);
    realloc — доля поездо-часов, перенесённых между группами относительно расписания;
    align   — корреляция «добавленные поезда ↔ ожидаемое наполнение по расписанию» (куда отправляются поезда).
    """
    G, N = env.G, env.N
    full = rollout(env, policy, days, record=True)
    g_full = float(np.mean(1 - full["J"] / full["J_sched"]))

    def with_override(f):
        r = rollout(env, policy, days, override=f)
        return float(np.mean(1 - r["J"] / r["J_sched"]))

    loo, solo = {}, {}
    for i in range(N):
        def drop_i(e, a, i=i):
            a = a.clone()
            a[:, i] = A_SCHED
            return a

        def only_i(e, a, i=i):
            b = torch.full_like(a, A_SCHED)
            b[:, i] = a[:, i]
            return b

        nm = GROUPS[env.P.groups[i]][0] if i < G else "координатор"
        loo[nm] = g_full - with_override(drop_i)
        solo[nm] = with_override(only_i)
    tr = full["trace"]
    n = tr["n"]                                                    # (B, 24, G)
    bi = torch.as_tensor(days)
    s = env.sched[bi].numpy()
    rtt = env.rtt.numpy()
    dev = (n - s) * rtt
    realloc = float((np.abs(dev).sum(-1) / 2).sum(-1).mean() / env.budget[bi].numpy().mean())
    rho_s = (env.kappa.numpy() * env.dem[bi].numpy() / np.maximum(s, 1e-3))
    x, y = (n / np.maximum(s, 1e-3) - 1).ravel(), rho_s.ravel()
    ok = s.ravel() > 0
    align = float(np.corrcoef(x[ok], y[ok])[0, 1]) if x[ok].std() > 1e-9 else 0.0
    return {"policy": name, "gain": g_full, "synergy": g_full - sum(solo.values()), "realloc": realloc,
            "align": align, **{f"loo|{k}": v for k, v in loo.items()}, **{f"solo|{k}": v for k, v in solo.items()}}


# =============================================================================
# Стресс-тест (PROTOCOL.md разд. 9)
# =============================================================================
STRESS = {
    "S1_surge": "непредвиденный всплеск: +40% спроса на одной группе 17–21 ч (прогноз не знает)",
    "S1b_surge_known": "то же, но всплеск есть в прогнозе (плановое событие)",
    "S2_storm": "непогода: −40% спроса весь день (прогноз не знает)",
    "S3_fleet": "нехватка подвижного состава: парк −15%, бюджет поездо-часов −10%",
    "S4_fc_bias": "сбой прогноза: прогноз занижен на 25%",
    "S5_calib": "ошибка калибровки: реальное наполнение на 25% выше принятого",
}


def stress(data: EnvData, P: EnvParams, name: str, days, seed=0):
    """Возвращает (EnvData, EnvParams) со стрессом на сутках days. Обучение не повторяется."""
    rng = np.random.default_rng(seed)
    d = replace(data, dem=data.dem.copy(), fc={k: v.copy() for k, v in data.fc.items()})
    P2 = P
    G = data.G
    if name.startswith("S1"):
        gs = rng.integers(0, G, len(days))
        for di, g in zip(days, gs):
            d.dem[di, 17:21, g] *= 1.4
            if name == "S1b_surge_known":
                for k, fc in d.fc.items():
                    for h in range(H):
                        lo, hi = max(17 - h, 0), max(21 - h, 0)
                        if hi > 0:
                            fc[di, h, lo:hi, g] *= 1.4
    elif name == "S2_storm":
        d.dem[days] *= 0.6
    elif name == "S3_fleet":
        P2 = replace(P, fleet_scale=0.85, budget_scale=0.90)
    elif name == "S4_fc_bias":
        for k in d.fc:
            d.fc[k][days] *= 0.75
    elif name == "S5_calib":
        P2 = replace(P, kappa_scale=1.25)
    else:
        raise KeyError(name)
    return d, P2


# =============================================================================
# Обучение одного прогона и пакетный запуск
# =============================================================================
def results_dir() -> Path:
    p = Path("/kaggle/working/results") if Path("/kaggle/working").exists() else Path("results")
    p.mkdir(parents=True, exist_ok=True)
    return p


def run_job(job: dict) -> dict:
    """job: algo, seed, budget, data_path, P (dict), cfg (dict), tag, parts, out. Пишет чекпойнт, кривые и суточные метрики."""
    import marl_algos as ma
    torch.set_num_threads(1)
    out = Path(job["out"])
    out.mkdir(parents=True, exist_ok=True)
    tag = job["tag"]
    done_flag = out / f"{tag}.done.json"
    if done_flag.exists() and not job.get("force"):
        return json.loads(done_flag.read_text())
    t0 = time.perf_counter()
    try:
        data = EnvData.load(job["data_path"])
        P = EnvParams(**job.get("P", {}))
        env = MetroEnv(data, P)
        venv = MetroEnv(data, P)
        va = data.idx("val")
        learner = ma.make(job["algo"], env, job["seed"], **job.get("cfg", {}))
        eval_fn = lambda pol: float(rollout(venv, pol, va)["reward"].mean())
        logs = []
        learner.train(env, data.idx("train"), job["budget"], eval_fn, n_evals=job.get("n_evals", 12),
                      log=logs.append, time_limit=job.get("time_limit"))
        torch.save({"state": learner.state_dict(), "cfg": learner.cfg, "P": P.__dict__, "algo": job["algo"],
                    "seed": job["seed"]}, out / f"{tag}.pt")
        hist = pd.DataFrame(learner.history).assign(tag=tag, algo=job["algo"], seed=job["seed"])
        hist.to_csv(out / f"{tag}.curve.csv", index=False)
        frames = []
        for part in job.get("parts", ["val", "test", "holdout"]):
            days = data.idx(part)
            if len(days):
                frames.append(evaluate(learner, venv, days, job.get("name", job["algo"]), job["seed"], part))
        daily = pd.concat(frames, ignore_index=True)
        daily["tag"] = tag
        daily.to_parquet(out / f"{tag}.daily.parquet", index=False)
        res = {"tag": tag, "algo": job["algo"], "name": job.get("name", job["algo"]), "seed": job["seed"],
               "budget": job["budget"], "steps": learner.steps, "best_step": learner.best_step,
               "best_val_reward": learner.best_val, "train_sec": learner.train_sec,
               "total_sec": time.perf_counter() - t0, "n_params": learner.n_params(),
               "cfg": json.dumps(learner.cfg, ensure_ascii=False), "P": P.tag(),
               "time": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "ok": True}
    except Exception as e:  # noqa: BLE001
        res = {"tag": tag, "algo": job["algo"], "seed": job["seed"], "ok": False, "error": repr(e),
               "trace": traceback.format_exc()[-2000:]}
    done_flag.write_text(json.dumps(res, ensure_ascii=False, default=str))
    return res


def run_many(jobs: list, workers: int = None, log=print) -> pd.DataFrame:
    """Параллельный запуск (процессы, по 1 потоку torch на процесс). Журнал — runs.csv."""
    import multiprocessing as mp
    workers = workers or max(1, (os.cpu_count() or 2))
    t0 = time.perf_counter()
    res = []
    if workers == 1:
        for j in jobs:
            r = run_job(j)
            res.append(r)
            log(f"  {r['tag']}: {'ок' if r.get('ok') else 'ОШИБКА ' + r.get('error', '')} "
                f"{r.get('train_sec', 0):.0f} c")
    else:
        ctx = mp.get_context("fork" if os.name != "nt" else "spawn")
        with ctx.Pool(workers, maxtasksperchild=1) as pool:
            for r in pool.imap_unordered(run_job, jobs):
                res.append(r)
                log(f"  [{len(res)}/{len(jobs)} {time.perf_counter() - t0:.0f} c] {r['tag']}: "
                    f"{'ок' if r.get('ok') else 'ОШИБКА ' + r.get('error', '')} {r.get('train_sec', 0):.0f} c")
    df = pd.DataFrame(res)
    f = results_dir() / "runs.csv"
    df.to_csv(f, mode="a", header=not f.exists(), index=False)
    return df


def find_all(pattern: str, roots=("/kaggle/input", "/kaggle/working/results", "results")) -> list[Path]:
    """Все файлы по шаблону во всех подключённых выводах ноутбуков (и локально). Дубликаты по имени — первый."""
    seen, out = set(), []
    for r in roots:
        if Path(r).exists():
            for p in sorted(Path(r).rglob(pattern)):
                if p.name not in seen:
                    seen.add(p.name)
                    out.append(p)
    return out


def load_runs(pattern="*.daily.parquet", roots=("/kaggle/input", "/kaggle/working/results", "results")) -> pd.DataFrame:
    fs = find_all(pattern, roots)
    return pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def load_policy(ckpt: Path, env: MetroEnv):
    import marl_algos as ma
    ck = torch.load(ckpt, weights_only=False)
    learner = ma.make(ck["algo"], env, ck["seed"], **{k: v for k, v in ck["cfg"].items()})
    learner.load_state_dict(ck["state"])
    return learner


def baseline_frames(data: EnvData, P: EnvParams, parts=("val", "test", "holdout"), mpc=("m4w", "norm", "hyb", "oracle")):
    """Расписание и MPC-планировщики (детерминированы, seed не нужен)."""
    env = MetroEnv(data, P)
    frames = []
    pols = [SchedulePolicy()] + [MPCPolicy(s) for s in mpc if s == "oracle" or s in data.fc]
    for part in parts:
        days = data.idx(part)
        if not len(days):
            continue
        for pol in pols:
            if pol.name == "mpc_hyb":
                ok = ~np.isnan(data.fc["hyb"][days, 0, 0, 0])
                dd = days[ok]
                if not len(dd):
                    continue
                frames.append(evaluate(pol, env, dd, pol.name, 0, part))
            else:
                frames.append(evaluate(pol, env, days, pol.name, 0, part))
    return pd.concat(frames, ignore_index=True)

In [ ]:
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
sys.path.insert(0, str(HERE))
import marl_data as md
import marl_env as me
from marl_eval import to_md as me_md
OUT = HERE / "results"
OUT.mkdir(parents=True, exist_ok=True)
LOG = []
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    LOG.append(s)
t0 = time.time()
root = md.find_vkr_root()
y, cov, static = md.load_vkr(root)
log("датасет ВКР:", root, "| входы", y.shape, "|", y.index.min(), "…", y.index.max())

## 1. Свежий отложенный период (holdout): входы после конца датасета ВКР

In [ ]:
vkr_end = y.index.max()
start = (vkr_end + pd.Timedelta(hours=1)).normalize()
end = pd.Timestamp.now().normalize()
try:
    fresh = md.download_ridership(start, end, list(y.columns), log=log)
    fresh = fresh[fresh.index >= start]
    if len(fresh):
        per_day = fresh.groupby(fresh.index.normalize()).size()
        full = per_day[per_day >= 23].index
        last_full = full.max() if len(full) else None
        # день считается полным, если есть ≥ 23 часа и сумма входов не меньше 60% медианы тех же дней недели в 2026
        tot = fresh.sum(axis=1).groupby(fresh.index.normalize()).sum()
        ref = y[y.index >= "2026-06-01"].sum(axis=1).groupby(y[y.index >= "2026-06-01"].index.normalize()).sum()
        ref_dow = ref.groupby(ref.index.dayofweek).median()
        ok_days = [d for d in full if tot[d] >= 0.6 * ref_dow[d.dayofweek]]
        # отложенный период — непрерывный отрезок от начала до первого неполного дня
        cont = []
        for d in pd.date_range(start, end, freq="D"):
            if d in ok_days:
                cont.append(d)
            else:
                break
        fresh = fresh[fresh.index.normalize().isin(cont)]
        log(f"свежие входы: {len(cont)} полных суток подряд, {cont[0].date() if cont else '—'} … "
            f"{cont[-1].date() if cont else '—'}; станций с данными {int((fresh.sum() > 0).sum())}")
    fresh.to_parquet(OUT / "holdout_ridership.parquet")
except Exception as e:  # noqa: BLE001
    fresh = pd.DataFrame(columns=y.columns)
    log("ОШИБКА загрузки свежих входов — holdout будет пустым, записать в отступления:", repr(e))
y_all = pd.concat([y, fresh.astype("float32")]) if len(fresh) else y
holdout_end = y_all.index.max().normalize()

## 2. Расписания MTA: отправления по часам, проверка дубликатов, время рейса

In [ ]:
t1 = time.time()
trips = md.download_trips(pd.Timestamp("2022-12-31"), holdout_end + pd.Timedelta(days=1), log=log)
trips.to_parquet(OUT / "trips.parquet", index=False)
log(f"расписание: {len(trips):,} строк, форма запроса «{trips.attrs.get('query', '?')}», {time.time() - t1:.0f} c")
tt = trips.groupby("trip_type")["n"].sum()
log("рейсы по trip_type:", tt.to_dict())
# если обычные рейсы (trip_type = 0) — подавляющее большинство, прочие (служебные) не считаем
if "0" in tt and tt["0"] / tt.sum() > 0.9:
    trips = trips[trips["trip_type"] == "0"]
    log("оставлены рейсы trip_type = 0")
samples = pd.to_datetime(["2023-03-08", "2024-07-13", "2025-10-08", "2026-02-11", "2026-05-16", "2026-08-12"])
try:
    dup = md.check_trip_duplicates(samples, log=log)
except Exception as e:  # noqa: BLE001
    dup = pd.DataFrame(columns=["date", "rows", "unique_trips", "dup_share", "supplements"])
    log("проверка дубликатов рейсов не выполнена (записать в паспорт):", repr(e)[:400])
dup.to_csv(OUT / "trip_duplicates_check.csv", index=False)
if len(dup) and dup["dup_share"].max() > 0.01:
    log("ВНИМАНИЕ: в расписании есть дубликаты рейсов > 1% — записать в паспорт как проблему качества")
q_dates = []
for yy in [2023, 2024, 2025, 2026]:
    for mm in [1, 4, 7, 10]:
        first = pd.Timestamp(f"{yy}-{mm:02d}-01")
        if first > holdout_end:
            continue
        wed = first + pd.Timedelta(days=(2 - first.dayofweek) % 7 + 7)      # 2-я среда месяца
        sat = first + pd.Timedelta(days=(5 - first.dayofweek) % 7 + 7)      # 2-я суббота месяца
        q_dates += [wed, sat]
try:
    durations = md.download_durations([d for d in q_dates if d <= holdout_end], log=log)
    durations.to_parquet(OUT / "durations.parquet", index=False)
    log(f"время рейса: {len(durations):,} рейсов, {durations['date'].nunique()} дат")
except Exception as e:  # noqa: BLE001
    durations = None
    log(f"время рейса не скачано — время оборота по умолчанию {md.RTT_DEFAULT_H} ч (записать в паспорт):", repr(e)[:400])

## 3. Прогноз гибрида ВКР (если подключён вывод ноутбука 16)

In [ ]:
hyb = {}
parts = []
for part in ["val", "test"]:
    f = md.find_file(f"hyb_blend__{part}.npz")
    if f is None:
        continue
    z = np.load(f, allow_pickle=False)
    assert (z["stations"].astype(str) == np.array(y.columns)).all(), "порядок станций в прогнозе ВКР не совпадает"
    parts.append((pd.DatetimeIndex(z["origins"]), z["yhat"].astype("float32")))
    log(f"прогноз гибрида ВКР {part}: точек {len(z['origins'])}, {f}")
if parts:
    hyb["hyb"] = (pd.DatetimeIndex(np.concatenate([p[0] for p in parts])), np.concatenate([p[1] for p in parts]))
else:
    log("прогноз гибрида ВКР не подключён — MPC «гибрид ВКР» и H6(а) будут без него (записать)")

## 4. Массивы среды

In [ ]:
data, pas = md.build_env_data(y_all, cov, static, trips, durations, hyb=hyb, log=log)
data.save(OUT / "env_data.npz")
log("сутки:", pas["days"], "| период", pas["period"])
log("время оборота, ч:", pas["rtt_h"])
log("парк (поездов в работе):", pas["fleet_max_trains"], "| бюджет буднего дня, поездо-ч:",
    round(pas["budget_trainh_median_weekday"]))

## 5. Паспорт данных: описательная статистика и проблемы качества

In [ ]:
G = data.G
names = [g for g, _ in me.GROUPS]
desc = []
for p in ["train", "val", "test", "holdout"]:
    m = data.part == p
    if not m.any():
        continue
    for g in range(G):
        dd, ss = data.dem[m][..., g], data.sched[m][..., g]
        desc.append({"часть": p, "группа": names[g], "сутки": int(m.sum()),
                     "входов в сутки (медиана)": int(np.median(dd.sum(1))),
                     "пик часа (95-й перц.)": int(np.percentile(dd, 95)),
                     "поездов в час (среднее)": round(float(ss.mean()), 1),
                     "поездов в час (макс.)": round(float(ss.max()), 1),
                     "κ·спрос/поезд, пик 95%": round(float(np.percentile(data.kappa[g] * dd / np.maximum(ss, 1e-3), 95)), 2)})
desc = pd.DataFrame(desc)
desc.to_csv(OUT / "describe_groups.csv", index=False)
print(desc.to_string(index=False))
fw = pd.Series(pas["forecast_wape"]).rename("WAPE")
fw.index = pd.MultiIndex.from_tuples([tuple(i.split("|")) for i in fw.index], names=["прогноз", "часть", "горизонт"])
fw = fw.unstack("часть")
fw.to_csv(OUT / "forecast_wape.csv")
print(fw.round(4).to_string())
flags = pd.DataFrame({k: v for k, v in data.flags.items() if k != "daytype"}, index=pd.DatetimeIndex(data.day))
flags["part"] = np.where(data.part == "", "история", data.part)
fl = flags.groupby("part").mean(numeric_only=True).round(3)
print(fl.to_string())

In [ ]:
issues = [
    ("часы без данных (заполнены интерполяцией)", pas["hours_filled"], "весенний перевод часов и пропуски выгрузки"),
    ("станций без управляемых маршрутов", pas["stations_no_group"], "только челноки — входы выпадают"),
    ("доля входов на таких станциях", round(pas["share_entries_no_group"], 4), ""),
    ("доля входов в часы без работающих групп станции", round(pas["share_entries_no_service"], 4),
     "станция обслуживается только закрытыми линиями (плановые работы)"),
    ("сутки без расписания (подставлено типичное)", pas["days_schedule_filled"], ", ".join(pas["days_schedule_filled_list"][:10])),
    ("неизвестные маршруты в расписании", len(pas["routes_unknown"]), ", ".join(pas["routes_unknown"])),
    ("сутки без ковариат ВКР (календарь по правилам)", pas["days_without_vkr_covariates"], "holdout после 24.09.2026"),
    ("дубликаты рейсов в образцах расписания", round(float(dup["dup_share"].max()) if len(dup) else float("nan"), 4), ""),
]
lines = ["# ЛР3. Паспорт данных", "",
         f"Сгенерировано ноутбуком lab3_00_data {pd.Timestamp.now():%d.%m.%Y}.", "",
         "## 1. Источники", "",
         "| источник | что берём | период |", "| --- | --- | --- |",
         f"| датасет ВКР `VKR_PM232_Pevin_MTA` (MTA Subway Hourly Ridership `wujg-7c2s`, `5wq4-mkjj`) | почасовые входы 424 комплексов, календарь | {y.index.min():%d.%m.%Y} – {y.index.max():%d.%m.%Y} |",
         f"| MTA Subway Hourly Ridership `5wq4-mkjj` (скачано здесь) | свежие входы для отложенного периода | {start:%d.%m.%Y} – {holdout_end:%d.%m.%Y} |",
         "| MTA Subway Schedules 2023–2026 (`7pnn-mafy`, `ebrw-j62c`, `q9nv-uegs`, `g8es-h7gb`) | отправления с начальной станции по часам, время рейса | 2023–2026 |",
         "| гибрид ВКР `hyb_blend` (вывод ноутбука 16 ВКР) | прогноз на 24 ч по станциям | " + ("val, test" if hyb else "не подключён") + " |",
         "", "## 2. Разбиение (строго по времени)", "",
         "| часть | сутки |", "| --- | --- |"] + [f"| {k} | {v} |" for k, v in pas["days"].items()] + [
         "", f"Период массивов: {pas['period'][0]} – {pas['period'][1]}; первые 4 недели — только история для прогноза.",
         "", "## 3. Описательная статистика по группам линий", "", me_md(desc),
         "", "## 4. Калибровка и ресурсы", "",
         f"- Час пик (будни, train) по группам: {pas['peak_hour']}",
         f"- Медианный спрос в час пик: {pas['peak_demand']}",
         f"- Поездов в час пик по расписанию: {pas['peak_tph']}",
         f"- Время оборота, ч: {pas['rtt_h']} ({pas['rtt_source']})",
         f"- Парк (поездов в работе одновременно): {pas['fleet_max_trains']}; бюджет буднего дня: {round(pas['budget_trainh_median_weekday'])} поездо-ч",
         "", "## 5. Качество прогнозов спроса групп (WAPE, группа × час)", "", me_md(fw.round(4).reset_index()),
         "", "## 6. Режимы суток (доля)", "", me_md(fl.reset_index()),
         "", "## 7. Проблемы качества", "", "| проблема | значение | комментарий |", "| --- | --- | --- |"] + \
        [f"| {a} | {b} | {c} |" for a, b, c in issues] + [
         "", "## 8. Смещения выборки (аналог survivorship bias)", "",
         "- Состав станций фиксирован (424 комплекса ВКР) на весь период; открытые позже станции не появляются, "
         "закрытые на ремонт остаются с нулевыми входами — их спрос не перераспределяется на соседние станции.",
         "- Расписание с поправками отражает плановые работы, но не сбои в реальном времени: фактическое число поездов "
         "в день сбоя ниже расписания, а в среде — нет.",
         "- Спрос — входы: пересадки внутри системы не учитываются; нагрузка магистралей с пересадочными потоками занижена "
         "и компенсируется только калибровкой κ.",
         "", "## 9. Схема предобработки", "",
         "1. Полная сетка часов; пропуски — линейная интерполяция (≤ 6 ч).",
         "2. Станция → группы линий по названию комплекса; входы делятся пропорционально поездам по расписанию.",
         "3. Расписание: отправления с начальной станции (stop_order = 1) по часу отправления, среднее двух направлений; "
         "сутки без расписания — типичное расписание типа дня по train.",
         "4. Прогнозы m4w и norm_level по ряду группы (коэффициенты norm_level — только train); гибрид ВКР — сумма прогнозов станций.",
         "5. Калибровка κ, нормировки, парк и пропускная способность — только по train (кроме пропускной способности — максимум расписания).",
         ]
(OUT / "data_passport.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:40]))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
tr = data.part == "train"
wk = tr & (data.flags["daytype"] == 0)
for g, ax in enumerate(axes.ravel()):
    ax.plot(np.median(data.dem[wk][..., g], 0), label="спрос, будни (медиана)", color="#1f77b4")
    ax2 = ax.twinx()
    ax2.plot(np.median(data.sched[wk][..., g], 0), color="#d62728", ls="--", label="поездов/ч")
    ax.set_title(names[g])
    ax.set_xticks([0, 6, 12, 18, 23])
fig.suptitle("Спрос группы (входов/ч, синий) и расписание (поездов/ч, красный пунктир), будни train")
fig.tight_layout()
fig.savefig(OUT / "fig_demand_vs_schedule.png", dpi=120)

## 6. Проверки среды
Расписание: награда ровно −24 и нулевой перерасход. MPC и оракул на val — ориентир для обучения.

In [ ]:
env = me.MetroEnv(data, me.EnvParams())
va = data.idx("val")
chk = []
for pol in [me.SchedulePolicy(), me.MPCPolicy("m4w"), me.MPCPolicy("norm"), me.MPCPolicy("oracle")]:
    r = me.rollout(env, pol, va)
    g = 1 - r["J"] / r["J_sched"]
    chk.append({"политика": pol.name, "награда": r["reward"].mean(), "выигрыш, %": 100 * g.mean(),
                "перерасход max": r["overrun"].max(), "поездо-ч, % бюджета": 100 * r["used_frac"].mean()})
chk = pd.DataFrame(chk)
print(chk.round(3).to_string(index=False))
assert abs(chk.iloc[0]["награда"] + 24) < 1e-3, "расписание должно давать −24"
chk.to_csv(OUT / "env_checks_val.csv", index=False)
(OUT / "lab3_00.log").write_text("\n".join(LOG), encoding="utf-8")
print(f"готово за {(time.time() - t0) / 60:.1f} мин")